# Public support for plastic policy in Nigeria and Ghana: full analysis

Article: *What explains public support for plastic policy? A comparative study of Nigeria and Ghana* (Abdulwakil Olawale Saba, Mutala Mohammed, Musa Adamu Ibrahim, Abdulwaheed Tella, Dokuboba Amachree).

This notebook reproduces **every table and figure** in the article. It starts from the de-identified merged file `data/processed/survey_merged_deidentified.csv`, or from the two raw Google Forms exports when they are present (they are not public; see `data/raw/README.md`).

**What it does, in order**

| Step | What happens | Output |
|---|---|---|
| 1 | Load the merged file, or (with the raw exports) read the two survey files, give the questions short names, recode answers to numbers and **merge** the two countries into one file | `data/processed/survey_merged_deidentified.csv` (+ `.xlsx`) |
| 2 | Describe the samples and compare every item between countries | Tables T0-T3, Fig. 1, Fig. 2, Fig. S1 |
| 3 | Latent class analysis (R) to find attitude segments and response styles | Tables T10a-T10i |
| 4 | Measurement model, invariance tests and multi-group SEM (R) | Tables T4-T7 |
| 5 | Machine learning: prediction, transfer between countries and SHAP | Tables T8-T9 |
| 6 | Coding of the open-ended answers | Table T11 |
| 7 | Remaining figures | Fig. 3-7, Fig. S2-S4 |
| 8 | All tables in one Excel workbook | `outputs/All_tables.xlsx` |

**How to run**

1. Install Python packages (Python 3.12 or later): `pip install -r requirements.txt`.
2. Install R (4.3 or later). The notebook installs the R packages it needs into `r_library/` by itself.
3. In Jupyter, choose **Kernel > Restart & Run All** (or **Run All** in VS Code).
4. Optional: run `python compare_tables.py --snapshot` before and `python compare_tables.py` after, to check your results against the published tables.

A full run takes about 1.5 to 2 hours on a laptop, mostly in the SEM robustness checks and the machine-learning bootstraps. Set `QUICK = True` in the next cell for a 10-minute test run with fewer bootstrap draws. Quick-run numbers will differ slightly from the paper.

In [ ]:
# ---------------------------------------------------------------- settings you may change
QUICK = False        # True = fast test run with fewer bootstrap draws (numbers differ slightly from the paper)
RSCRIPT = None       # None = find R by itself; or give the full path, e.g. r"C:\Program Files\R\R-4.3.3\bin\Rscript.exe"

# ---------------------------------------------------------------- libraries
import os, re, json, glob, shutil, subprocess, warnings
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.lines import Line2D
from matplotlib.patches import FancyBboxPatch, Ellipse, FancyArrowPatch
import geopandas as gpd
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.proportion import proportion_confint
from statsmodels.nonparametric.smoothers_lowess import lowess
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss, average_precision_score
from sklearn.model_selection import GridSearchCV, RepeatedStratifiedKFold, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import shap
warnings.filterwarnings("ignore")

# ---------------------------------------------------------------- folders
def find_project_folder():
    # The project folder is the one holding data/processed/survey_merged_deidentified.csv.
    # It is found by itself, so nothing needs to be typed, wherever Jupyter or VS Code was opened.
    marker = Path("data") / "processed" / "survey_merged_deidentified.csv"
    starts = [Path.cwd()]
    if "__vsc_ipynb_file__" in globals():                 # VS Code knows the notebook's own path
        starts.insert(0, Path(globals()["__vsc_ipynb_file__"]).parent)
    if os.environ.get("JPY_SESSION_NAME"):               # so does recent Jupyter
        starts.insert(0, Path(os.environ["JPY_SESSION_NAME"]).parent)
    for s in starts:
        for p in [s, *s.parents][:6]:
            for cand in (p, p / "Plastic_NG_GH_project"):
                if (cand / marker).exists() or (cand / "Plastic_NG_GH_analysis.ipynb").exists():
                    return cand.resolve()
    raise FileNotFoundError("Open this notebook from inside the project folder (the one that contains data/processed/).")

def find_rscript(given=None):
    # Finds Rscript on PATH or in the usual install folders (Windows, macOS, Linux).
    if given:
        if Path(given).exists() or shutil.which(given):
            return str(given)
        raise FileNotFoundError(f"RSCRIPT = {given!r} does not exist. Set RSCRIPT = None to search automatically.")
    if shutil.which("Rscript"):
        return shutil.which("Rscript")
    pats = [r"C:\Program Files\R\R-*\bin\Rscript.exe", r"C:\Program Files\R\R-*\bin\x64\Rscript.exe",
            os.path.expandvars(r"%LOCALAPPDATA%\Programs\R\R-*\bin\Rscript.exe"),
            "/Library/Frameworks/R.framework/Resources/bin/Rscript", "/opt/homebrew/bin/Rscript",
            "/usr/local/bin/Rscript", "/usr/bin/Rscript"]
    found = [f for p in pats for f in glob.glob(p)]
    if found:
        return sorted(found, key=lambda f: [int(x) for x in re.findall(r"\d+", f)])[-1]   # newest R
    raise FileNotFoundError("R was not found. Install R from https://cran.r-project.org, then run this cell again.")

ROOT = find_project_folder()
os.chdir(ROOT)                       # every relative path below now points inside the project folder
RSCRIPT = find_rscript(RSCRIPT)
RAW, PROC, GEO = ROOT / "data" / "raw", ROOT / "data" / "processed", ROOT / "data" / "geo"
TAB, FIG, MOD, RDIR = ROOT / "outputs" / "tables", ROOT / "outputs" / "figures", ROOT / "outputs" / "models", ROOT / "r_scripts"
for p in (PROC, TAB, FIG, MOD, RDIR, ROOT / "docs", ROOT / "outputs" / "logs"):
    p.mkdir(parents=True, exist_ok=True)
DATA = PROC / "survey_merged_deidentified.csv"

SEED = 20231101                      # one seed for everything, so results are identical on every run
N_BOOT_AUC = 300 if QUICK else 2000  # bootstrap draws for AUC confidence intervals
N_BOOT_SHAP = 20 if QUICK else 100   # bootstrap refits for SHAP
N_BOOT_RB = 300 if QUICK else 2000   # bootstrap draws for item effect sizes
EQ_MARGIN = 0.05   # equivalence margin for transfer gaps (AUC units)
COUNTRIES = ["Nigeria", "Ghana"]
COL = {"Nigeria": "#1b7a4a", "Ghana": "#c9822b"}   # country colours used in every figure
print("Project folder:", ROOT)
print("R:", RSCRIPT)

In [ ]:
# ---------------------------------------------------------------- figure style (600 dpi, no overlaps)
mpl.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 8, "axes.titlesize": 8.5, "axes.labelsize": 8,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7, "axes.spines.top": False,
    "axes.spines.right": False, "axes.linewidth": 0.6, "savefig.dpi": 600, "figure.dpi": 110,
    "pdf.fonttype": 42, "legend.frameon": False})

def savefig(fig, name):
    # Every figure is saved three ways: 600-dpi PNG, 600-dpi TIFF (for journals) and vector PDF.
    for ext, kw in [("png", {}), ("pdf", {}), ("tif", {"pil_kwargs": {"compression": "tiff_lzw"}})]:
        fig.savefig(FIG / f"{name}.{ext}", dpi=600, bbox_inches="tight", pad_inches=0.05, **kw)
    plt.show()

def run_r(script):
    # Runs one R script (from r_scripts/, or a full path), shows its output as it runs and keeps a log.
    path = Path(script) if Path(script).is_absolute() else RDIR / script
    log = ROOT / "outputs" / "logs" / f"{path.name}.log"
    lines = []
    with subprocess.Popen([RSCRIPT, str(path), str(ROOT)], cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding="utf-8", errors="replace") as proc:
        for line in proc.stdout:
            lines.append(line)
            if not line.lstrip().startswith(("Warning", "lavaan->", "The variance", "appear to be", "is close to",
                                             "some cases", "cases were", "variable(s)")):
                print(line, end="")
    log.write_text("".join(lines), encoding="utf-8")
    if proc.returncode != 0:
        raise RuntimeError(f"{path.name} stopped with an error (see the lines above, or {log.relative_to(ROOT)})")


### R packages (checked once, at the start)

The cell below writes `r_scripts/00_setup.R`, which every R script loads first. It finds the project folder by itself and installs any missing R package as a ready-made binary into `r_library/` inside this folder, so you do not need administrator rights or Rtools. The first run needs an internet connection and takes a few minutes; later runs skip it.

In [ ]:
%%writefile r_scripts/00_setup.R
# Shared setup for every R script in this project. Nothing here needs editing.
#
# 1. Finds the project folder by itself (the folder that holds data/processed/survey_merged_deidentified.csv),
#    whether the script is run from the notebook, with Rscript, or with source() / Run in RStudio.
# 2. Uses a package library inside the project folder (r_library/), so no administrator rights are needed
#    and everything stays in the one folder.
# 3. Installs any missing package as a ready-made CRAN binary (no Rtools or compiler needed on Windows).

DATA_FILE <- file.path("data", "processed", "survey_merged_deidentified.csv")
CRAN <- "https://cloud.r-project.org"
ALL_PACKAGES <- c("lavaan", "semTools", "psych", "GPArotation", "poLCA", "nnet")

find_project_root <- function() {
  starts <- character(0)
  given <- commandArgs(trailingOnly = TRUE)                        # folder passed by the notebook, if any
  if (length(given) && nzchar(given[1]) && dir.exists(given[1])) starts <- c(starts, given[1])
  file_arg <- gsub("~+~", " ", sub("^--file=", "", grep("^--file=", commandArgs(FALSE), value = TRUE)), fixed = TRUE)
  if (length(file_arg)) starts <- c(starts, dirname(file_arg[1]))
  for (i in rev(seq_len(sys.nframe()))) {                          # source("script.R")
    of <- sys.frame(i)$ofile
    if (!is.null(of)) starts <- c(starts, dirname(of))
  }
  p <- tryCatch(rstudioapi::getSourceEditorContext()$path, error = function(e) "")   # Run lines in RStudio
  if (length(p) == 1 && nzchar(p)) starts <- c(starts, dirname(p))
  starts <- c(starts, getwd())
  for (s in starts) {
    d <- normalizePath(s, winslash = "/", mustWork = FALSE)
    for (k in 1:6) {                                               # look in this folder and up to 5 above it
      if (file.exists(file.path(d, DATA_FILE))) return(d)
      if (file.exists(file.path(d, "Plastic_NG_GH_project", DATA_FILE))) return(file.path(d, "Plastic_NG_GH_project"))
      up <- dirname(d); if (up == d) break; d <- up
    }
  }
  stop("Could not find the project folder. Open or run this script from inside the project folder (the one that contains data/processed/).",
       call. = FALSE)
}

use_packages <- function(pkgs) {
  missing <- pkgs[!vapply(pkgs, requireNamespace, logical(1), quietly = TRUE)]
  if (length(missing)) {
    message("Installing missing R packages into ", LIB, ": ", paste(missing, collapse = ", "))
    type <- if (.Platform$OS.type == "windows" || Sys.info()[["sysname"]] == "Darwin") "binary" else "source"
    for (p in missing) {
      ok <- tryCatch({ suppressWarnings(install.packages(p, lib = LIB, repos = CRAN, type = type, quiet = TRUE))
                       requireNamespace(p, quietly = TRUE) }, error = function(e) FALSE)
      if (!isTRUE(ok) && type == "binary") {                       # no binary for this R version: try "both"
        ok <- tryCatch({ suppressWarnings(install.packages(p, lib = LIB, repos = CRAN, type = "both", quiet = TRUE))
                         requireNamespace(p, quietly = TRUE) }, error = function(e) FALSE)
      }
      if (!isTRUE(ok)) stop("Could not install the R package '", p, "'. Check your internet connection, then run ",
                            "install_R_packages.R once in R or RStudio and read the message it prints.", call. = FALSE)
    }
  }
  for (p in pkgs) suppressPackageStartupMessages(library(p, character.only = TRUE))
  invisible(TRUE)
}

root <- find_project_root()
LIB <- file.path(root, "r_library", paste0("R-", R.version$major, ".", sub("\\..*", "", R.version$minor)))
dir.create(LIB, recursive = TRUE, showWarnings = FALSE)
.libPaths(c(LIB, .libPaths()))
tab <- file.path(root, "outputs", "tables"); mod <- file.path(root, "outputs", "models")
for (p in c(tab, mod, file.path(root, "outputs", "logs"))) dir.create(p, recursive = TRUE, showWarnings = FALSE)


In [ ]:
run_r(ROOT / "install_R_packages.R")

## Step 1. Read, harmonise and merge the two surveys

The Nigerian and Ghanaian questionnaires are **identical, question for question** (48 columns in the same order). Only the country name in two questions and "state" versus "region" differ. Merging therefore means:

1. giving each of the 48 columns a short name (for example `flood`, `polluter_pays`);
2. turning the answers into numbers: *Strongly disagree* = 1 ... *Strongly agree* = 5, and *No* = 0, *Maybe* = 0.5, *Yes* = 1;
3. adding a `country` column and stacking the two files.

We also remove one Nigerian response submitted in November 2024, outside the June to December 2023 collection window, and we de-identify the data. Exact timestamps are reduced to the month, state/region of origin is dropped, free-text occupations are grouped, and e-mails, phone numbers and links are removed from the open answers.

If the raw exports are not in `data/raw/` (as in the public GitHub copy), these two cells simply load the merged file `data/processed/survey_merged_deidentified.csv`, and everything after Step 1 runs the same way.

In [ ]:
# Short names for the 48 columns, in questionnaire order
HARM = ["flood", "erosion", "vector", "infiltration", "buildings", "toxic", "microbes", "nutrients",
        "bags_billions", "waterways_tons", "aquatic", "food", "terrestrial", "harmful_env"]
ECO = HARM + ["trash_no_harm", "not_recycled", "not_reused", "charges"]
RESP = ["gov_resp", "everybody", "polluter_pays", "legal", "cheaper_plastic", "biodeg_alt", "cheap_bioplastic"]
SUPP = ["polluter_pays", "legal", "biodeg_alt", "cheap_bioplastic"]
LIKERT = ECO + RESP
YN = ["env_gathering", "heard_ce", "ce_reduce", "waste_wealth", "heard_bioplastic", "bioplastic_benefit"]
NAMES = (["timestamp", "gender", "age", "occupation", "marital", "education", "discipline",
          "env_gathering", "heard_ce", "ce_reduce", "waste_wealth", "wealth_text", "origin",
          "residence", "community", "residence_years", "income", "hh_size", "materials_used",
          "control_pref"] + ECO + ["sectors", "heard_bioplastic", "bioplastic_benefit"] + RESP)

LABEL = {  # plain-English labels used in tables and figures
    "flood": "Causes flooding", "erosion": "Causes erosion", "vector": "Breeding ground for pests and disease vectors",
    "infiltration": "Reduces water infiltration", "buildings": "Destroys buildings",
    "toxic": "Transports toxic chemicals", "microbes": "Harbours harmful microorganisms",
    "nutrients": "Reduces nutrient uptake by plants", "bags_billions": "Billions of bags discarded yearly",
    "waterways_tons": "Millions of tonnes reach waterways", "aquatic": "Harms aquatic animals",
    "food": "Plastic found in our food", "terrestrial": "Harms terrestrial animals",
    "harmful_env": "Harmful to the environment", "trash_no_harm": "Binning plastic will not harm environment",
    "not_recycled": "Plastic products are not recycled", "not_reused": "Plastic cannot be reused",
    "charges": "Extra charges on plastics reduce pollution", "gov_resp": "Management is government's responsibility",
    "everybody": "Everybody can play a role", "polluter_pays": "Polluter-pays principle", "legal": "Legal action",
    "cheaper_plastic": "Make plastic products cheaper", "biodeg_alt": "Provide biodegradable alternatives",
    "cheap_bioplastic": "Provide affordable bioplastics"}

LIK_CODE = {"Strongly disagree": 1, "Disagree": 2, "Neutral": 3, "Agree": 4, "Strongly agree": 5}
YN_CODE = {"Yes": 1.0, "Maybe": 0.5, "No": 0.0}

# Read both files and check they really have the same questions
# (the public GitHub copy has no raw files; the notebook then starts from the merged file)
HAVE_RAW = (RAW / "nigeria_raw.xlsx").exists() and (RAW / "ghana_raw.xlsx").exists()
if HAVE_RAW:
    ng_raw = pd.read_excel(RAW / "nigeria_raw.xlsx")
    gh_raw = pd.read_excel(RAW / "ghana_raw.xlsx")
    norm_q = lambda q, c, u: re.sub(r"\s+", "", q.replace(c, "X").replace(u, "S"))
    same = [norm_q(a, "Nigeria", "state") == norm_q(b, "Ghana", "region") for a, b in zip(ng_raw.columns, gh_raw.columns)]
    print(f"Nigeria: {ng_raw.shape}, Ghana: {gh_raw.shape}; identical questions: {sum(same)} of {len(same)}")
else:
    print("Raw exports not found in data/raw/. The next cell loads the merged, de-identified file instead.")

In [ ]:
def occ_group(s):
    # Groups free-text occupations into six broad categories
    s = str(s).strip().lower()
    if s in ("", "nan"):
        return np.nan
    for g, pat in [("Student", r"student|corps member|undergrad|postgrad"),
                   ("Academic or researcher", r"academ|research|lectur|teach|educat|phd"),
                   ("Public sector", r"civil|public|government|law enforcement"),
                   ("Business or self-employed", r"business|trader|entrepren|self|handwork|farmer|rider"),
                   ("Retired or unemployed", r"retire|unemploy")]:
        if re.search(pat, s):
            return g
    return "Other professional"

def redact(t):
    # Removes e-mail addresses, links and phone numbers from open answers
    if not isinstance(t, str):
        return t
    t = re.sub(r"\S+@\S+", "[email]", t)
    t = re.sub(r"https?://\S+|www\.\S+", "[url]", t)
    return re.sub(r"\+?\d[\d\s\-]{7,}\d", "[phone]", t).strip()

def merge_and_recode(ng_raw, gh_raw):
    # Renames, merges, recodes and de-identifies the two exports; saves the merged file and Table T0
    ng, gh = ng_raw.copy(), gh_raw.copy()
    ng.columns = gh.columns = NAMES
    ng.insert(0, "country", "Nigeria"); gh.insert(0, "country", "Ghana")
    late = ng.timestamp >= "2024-01-01"                    # one response outside the collection window
    flow = [{"step": "Responses exported", "Nigeria": len(ng), "Ghana": len(gh)},
            {"step": "Excluded: submitted after 31 Dec 2023", "Nigeria": int(late.sum()), "Ghana": 0}]
    d = pd.concat([ng[~late], gh], ignore_index=True)       # <- the merge
    d.insert(1, "resp_id", [f"{c[:2].upper()}{i:04d}" for i, c in enumerate(d.country, 1)])

    # Recode answers to numbers
    for c in LIKERT:
        d[c] = d[c].map(LIK_CODE)
    for c in YN:
        d[c] = d[c].map(YN_CODE)
    d["collection_month"] = d.timestamp.dt.to_period("M").astype(str)
    d["gender"] = d.gender.where(d.gender.isin(["Male", "Female"]))
    d["age"] = d.age.replace({"18 -27 years": "18-27", "28-37 years": "28-37", "38-47 years": "38-47", "More than 47 years": "48+"})
    d["marital"] = d.marital.replace({"Widow": "Widowed or divorced", "Divorced": "Widowed or divorced"})
    d["education"] = d.education.replace({"Primary": "Below tertiary", "Secondary": "Below tertiary", "Non-formal": "Below tertiary"})
    d["discipline"] = d.discipline.replace({"Veterinary Medicine": "Medical and Health Sciences", "Law": "Arts and Humanities"})
    d["occ_group"] = d.occupation.map(occ_group)
    d["community"] = d.community.replace({"Village/Suburb": "Village or suburb"})
    d["residence_years"] = d.residence_years.replace({"More than 10": ">10"})
    d["residence"] = d.residence.astype(str).str.strip().replace({"nan": np.nan})
    d["income_band"] = d.income   # currency-specific bands; only the top band is compared across countries
    d["income_top"] = d.income.astype(str).str.contains("More than").astype(float).where(d.income.notna())

    # Multi-select questions become 0/1 columns
    MATS = {"use_bag": "Polyethylene", "use_bottle": "Plastic bottle", "use_pvc": "PVC", "use_styrofoam": "Styreform"}
    PREFS = ["Recycle", "Reuse", "Replace", "Burning", "Burying"]
    SECS = {"sec_food_agric": "Food and agri", "sec_health": "Health care", "sec_sme": "Small and Medium",
            "sec_recreation": "Recreational", "sec_building": "Building", "sec_handwork": "Handwork", "sec_education": "Education"}
    for k, v in MATS.items():
        d[k] = d.materials_used.fillna("").str.contains(v, regex=False).astype(int)
    for m in PREFS:
        d["pref_" + m.lower()] = d.control_pref.fillna("").str.contains(m).astype(int)
    for k, v in SECS.items():
        d[k] = d.sectors.fillna("").str.contains(v, regex=False).astype(int)
    d.loc[d.materials_used.isna(), list(MATS)] = np.nan
    d.loc[d.control_pref.isna(), ["pref_" + m.lower() for m in PREFS]] = np.nan
    d.loc[d.sectors.isna(), list(SECS)] = np.nan

    # Engagement index (0-3): attended an environmental event + heard of circular economy + heard of bioplastic
    eng = d[["env_gathering", "heard_ce", "heard_bioplastic"]]
    d["engagement"] = eng.sum(axis=1, min_count=2) * 3 / eng.notna().sum(axis=1)

    # Response-quality flags (kept in the main analysis, dropped in a sensitivity check)
    lik = d[LIKERT]
    d["n_likert_answered"] = lik.notna().sum(axis=1)
    d["flag_no_likert"] = (d.n_likert_answered == 0).astype(int)
    d["flag_straightline"] = ((lik.nunique(axis=1) == 1) & (d.n_likert_answered >= 20)).astype(int)
    d["flag_harm_all_sd"] = (d[HARM] == 1).all(axis=1).astype(int)
    d["flag_inconsistent"] = (((d.harmful_env <= 1) & (d.aquatic >= 5) & (d.terrestrial >= 5)) |
                              ((d.harmful_env >= 5) & (d.aquatic <= 1) & (d.terrestrial <= 1))).astype(int)
    FLAGS = ["flag_no_likert", "flag_straightline", "flag_harm_all_sd", "flag_inconsistent"]
    d["flag_any_quality"] = d[FLAGS].max(axis=1)
    d["wealth_text"] = d.wealth_text.map(redact)
    for f in FLAGS + ["flag_any_quality"]:
        flow.append({"step": f"Quality flag (kept): {f}", "Nigeria": int(d.loc[d.country == "Nigeria", f].sum()),
                     "Ghana": int(d.loc[d.country == "Ghana", f].sum())})
    flow.append({"step": "Analytic sample", "Nigeria": int((d.country == "Nigeria").sum()), "Ghana": int((d.country == "Ghana").sum())})
    pd.DataFrame(flow).to_csv(TAB / "T0_data_flow.csv", index=False)

    KEEP = (["resp_id", "country", "collection_month", "gender", "age", "marital", "education", "discipline",
             "occ_group", "residence", "community", "residence_years", "income_band", "income_top", "hh_size"]
            + YN + LIKERT + list(MATS) + ["pref_" + m.lower() for m in PREFS] + list(SECS)
            + ["engagement", "wealth_text", "n_likert_answered"] + FLAGS + ["flag_any_quality"])
    d = d[KEEP]
    d.to_csv(DATA, index=False)
    d.to_excel(PROC / "survey_merged_deidentified.xlsx", index=False)
    print(pd.DataFrame(flow).to_string(index=False))
    print("Merged file:", DATA, d.shape)
    return d

if HAVE_RAW:
    d = merge_and_recode(ng_raw, gh_raw)
else:
    d = pd.read_csv(DATA)
    print("Loaded the merged file:", DATA, d.shape)

## Step 2. Who answered, and how do the countries differ item by item?

* **Table T1**: sample profile with chi-square tests and Cramér's V. Sparse tables use a permutation p-value.
* **Table T2**: the 25 agree/disagree items. Mann-Whitney U test and rank-biserial *r* (positive = higher in Ghana) with a bootstrap 95% CI. P-values are corrected for 25 tests with the Benjamini-Hochberg false discovery rate (*q*).
* **Table T3**: yes/no and tick-box questions, with Wilson 95% CIs and chi-square tests.

In [ ]:
def cramers_v(ct):
    chi2, p, dof, _ = stats.chi2_contingency(ct)
    return chi2, dof, p, np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1)))

PROFILE = {"gender": "Gender", "age": "Age group (years)", "marital": "Marital status", "education": "Highest education",
           "discipline": "Discipline", "occ_group": "Occupation", "community": "Settlement type",
           "residence_years": "Years in state/region", "income_top": "Highest income band", "hh_size": "Household size"}
rows = []
for v, lab in PROFILE.items():
    x = d[[v, "country"]].dropna().copy()
    if v == "income_top":
        x[v] = x[v].map({1.0: "Yes", 0.0: "No"})
    ct = pd.crosstab(x[v], x.country)[COUNTRIES]
    chi2, dof, p, cv = cramers_v(ct)
    if (stats.contingency.expected_freq(ct) < 5).any():      # sparse table: permutation p-value
        rng, sims = np.random.default_rng(1), []
        for _ in range(2000):
            sims.append(stats.chi2_contingency(pd.crosstab(x[v], rng.permutation(x.country.values)))[0])
        p = (np.sum(np.array(sims) >= chi2) + 1) / 2001
    for cat in ct.index:
        rows.append({"variable": lab, "category": cat,
                     "Nigeria_n": int(ct.loc[cat, "Nigeria"]), "Nigeria_pct": round(100 * ct.loc[cat, "Nigeria"] / ct["Nigeria"].sum(), 1),
                     "Ghana_n": int(ct.loc[cat, "Ghana"]), "Ghana_pct": round(100 * ct.loc[cat, "Ghana"] / ct["Ghana"].sum(), 1),
                     "chi2": round(chi2, 2), "df": dof, "p": round(p, 4), "cramers_v": round(cv, 3)})
T1 = pd.DataFrame(rows); T1.to_csv(TAB / "T1_sample_profile.csv", index=False)
T1.head(12)

In [ ]:
ngd, ghd = d[d.country == "Nigeria"], d[d.country == "Ghana"]
rows = []
for c in LIKERT:
    a, b = ngd[c].dropna(), ghd[c].dropna()
    u, p = stats.mannwhitneyu(b, a, alternative="two-sided")
    r_rb = 2 * u / (len(a) * len(b)) - 1                     # rank-biserial r, positive = Ghana higher
    rng = np.random.default_rng(7)
    boots = []
    for _ in range(N_BOOT_RB):
        aa, bb = rng.choice(a, len(a)), rng.choice(b, len(b))
        boots.append(2 * stats.mannwhitneyu(bb, aa).statistic / (len(a) * len(b)) - 1)
    rows.append({"item": c, "label": LABEL[c],
                 "block": "Harm perception" if c in HARM else ("Responsibility and instruments" if c in RESP else "Other beliefs"),
                 "Nigeria_n": len(a), "Nigeria_mean": a.mean(), "Nigeria_sd": a.std(), "Nigeria_pct_agree": 100 * (a >= 4).mean(),
                 "Ghana_n": len(b), "Ghana_mean": b.mean(), "Ghana_sd": b.std(), "Ghana_pct_agree": 100 * (b >= 4).mean(),
                 "U": u, "p": p, "r_rb": r_rb, "r_rb_lo": np.percentile(boots, 2.5), "r_rb_hi": np.percentile(boots, 97.5),
                 "missing_pct_NG": 100 * ngd[c].isna().mean(), "missing_pct_GH": 100 * ghd[c].isna().mean()})
T2 = pd.DataFrame(rows); T2["q_bh"] = multipletests(T2.p, method="fdr_bh")[1]
T2.round(4).to_csv(TAB / "T2_likert_item_comparison.csv", index=False)

BIN = {"env_gathering": "Attended environment-related gathering", "heard_ce": "Heard of circular economy",
       "ce_reduce": "Circular economy can reduce impacts (if heard)", "waste_wealth": "Plastic waste can generate wealth",
       "heard_bioplastic": "Heard of bioplastic", "bioplastic_benefit": "Bioplastics have economic/environmental benefits",
       "use_bag": "Uses polyethylene bags", "use_bottle": "Uses plastic bottles", "use_pvc": "Uses PVC",
       "use_styrofoam": "Uses Styrofoam", "pref_recycle": "Prefers recycling", "pref_reuse": "Prefers reuse",
       "pref_replace": "Prefers replacement", "pref_burning": "Prefers burning", "pref_burying": "Prefers burying",
       "sec_food_agric": "Sector: food and agriculture", "sec_health": "Sector: health care",
       "sec_sme": "Sector: small and medium businesses", "sec_recreation": "Sector: recreation",
       "sec_building": "Sector: building and plumbing", "sec_handwork": "Sector: handwork", "sec_education": "Sector: education"}
rows = []
for c, lab in BIN.items():
    a, b = (ngd[c].dropna() == 1).astype(int), (ghd[c].dropna() == 1).astype(int)   # "Yes" only
    chi2, p, _, _ = stats.chi2_contingency(np.array([[a.sum(), len(a) - a.sum()], [b.sum(), len(b) - b.sum()]]), correction=False)
    ca, cb = proportion_confint(a.sum(), len(a), method="wilson"), proportion_confint(b.sum(), len(b), method="wilson")
    diff = b.mean() - a.mean(); se = np.sqrt(a.mean() * (1 - a.mean()) / len(a) + b.mean() * (1 - b.mean()) / len(b))
    rows.append({"item": c, "label": lab, "Nigeria_n": len(a), "Nigeria_pct_yes": 100 * a.mean(), "NG_lo": 100 * ca[0],
                 "NG_hi": 100 * ca[1], "Ghana_n": len(b), "Ghana_pct_yes": 100 * b.mean(), "GH_lo": 100 * cb[0],
                 "GH_hi": 100 * cb[1], "diff_pp_GH_minus_NG": 100 * diff, "diff_lo": 100 * (diff - 1.96 * se),
                 "diff_hi": 100 * (diff + 1.96 * se), "chi2": chi2, "p": p})
T3 = pd.DataFrame(rows); T3["q_bh"] = multipletests(T3.p, method="fdr_bh")[1]
T3.round(4).to_csv(TAB / "T3_binary_item_comparison.csv", index=False)
T2[["label", "Nigeria_mean", "Ghana_mean", "r_rb", "q_bh"]].round(3)

### Figure 1: where respondents live

The maps show respondents per state (Nigeria) and region (Ghana). The bar charts beneath give the exact counts, so the maps need no text labels. The Natural Earth boundaries show Ghana's ten pre-2019 regions, so the three new regions in the data are drawn within their parent regions.

In [ ]:
geo = gpd.read_file(GEO / "admin1_ng_gh.geojson")
a0 = gpd.read_file(GEO / "admin0_region.geojson")
to_ne = {"FCT": "Federal Capital Territory", "Nasarawa": "Nassarawa", "Savannah": "Northern", "North East": "Northern",
         "Oti": "Volta", "Ahafo": "Brong Ahafo", "Bono": "Brong Ahafo", "Bono East": "Brong Ahafo", "Western North": "Western"}
cnt = (d.assign(res=d.residence.replace(to_ne)).groupby(["country", "res"]).size().rename("n").reset_index()
       .assign(adm0_a3=lambda x: x.country.map({"Nigeria": "NGA", "Ghana": "GHA"})))
assert set(cnt.res) <= set(geo.name), "a residence name did not match the map"
g = geo.merge(cnt, left_on=["adm0_a3", "name"], right_on=["adm0_a3", "res"], how="left")

fig = plt.figure(figsize=(7.2, 6.6), layout="constrained")
gsp = fig.add_gridspec(2, 2, height_ratios=[1.25, 1], width_ratios=[1.3, 1])
norm, cmap = LogNorm(vmin=1, vmax=200), "YlGnBu"
map_axes = []
for j, (code_, cty) in enumerate([("NGA", "Nigeria"), ("GHA", "Ghana")]):
    ax = fig.add_subplot(gsp[0, j]); map_axes.append(ax)
    sub = g[g.adm0_a3 == code_]
    a0[a0.adm0_a3 != code_].plot(ax=ax, color="#eeeeee", edgecolor="white", lw=0.4)
    sub[sub.n.isna()].plot(ax=ax, color="white", edgecolor="#9a9a9a", lw=0.3, hatch="////")
    sub[sub.n.notna()].plot(ax=ax, column="n", cmap=cmap, norm=norm, edgecolor="#555555", lw=0.3)
    x0, y0, x1, y1 = sub.total_bounds
    ax.set_xlim(x0 - 0.3, x1 + 0.3); ax.set_ylim(y0 - 0.3, y1 + 0.3); ax.set_aspect("equal")
    ax.set_title(f"{'ab'[j]}) {cty} (n = {(d.country == cty).sum()})", loc="left")
    ax.set_xlabel("Longitude (°E)"); ax.set_ylabel("Latitude (°N)")
    # bar chart of counts underneath (original names, so the new Ghanaian regions are shown separately)
    axb = fig.add_subplot(gsp[1, j])
    top = d[d.country == cty].residence.value_counts().head(10)[::-1]
    axb.barh(top.index, top.values, color=COL[cty], height=0.7)
    for yy, v in enumerate(top.values):
        axb.text(v + top.max() * 0.02, yy, str(v), va="center", fontsize=6.5)
    axb.set_xlim(0, top.max() * 1.18); axb.set_xlabel("Respondents")
    axb.set_title(f"{'cd'[j]}) Ten largest {'states' if cty == 'Nigeria' else 'regions'}, {cty}", loc="left")
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
cb = fig.colorbar(sm, ax=map_axes, location="bottom", shrink=0.45, aspect=30, pad=0.02)
cb.set_ticks([1, 3, 10, 30, 100, 200]); cb.set_ticklabels(["1", "3", "10", "30", "100", "200"]); cb.minorticks_off()
cb.set_label("Respondents per state or region (log scale); hatched = none")
savefig(fig, "Fig1_study_area")

### Figure 2: all 25 agree/disagree items, side by side

Each bar is centred on the middle of the *Neutral* category, so disagreement extends left and agreement extends right. Items are sorted within each block by the size of the country difference. Stars show FDR-adjusted significance.

In [ ]:
LIK_COLS = ["#b2182b", "#ef8a62", "#d9d9d9", "#67a9cf", "#2166ac"]
LIK_LABS = ["Strongly disagree", "Disagree", "Neutral", "Agree", "Strongly agree"]
BLOCKS = ["Harm perception", "Other beliefs", "Responsibility and instruments"]
star = lambda q: "***" if q < .001 else "**" if q < .01 else "*" if q < .05 else ""
order = {b: T2[T2.block == b].sort_values("r_rb").item.tolist() for b in BLOCKS}

fig = plt.figure(figsize=(7.2, 8.4), layout="constrained")
subfigs = fig.subfigures(1, 2, width_ratios=[1.75, 1], wspace=0.02)
for j, cty in enumerate(COUNTRIES):
    sf = subfigs[j]
    sf.suptitle(f"{cty} (n = {(d.country == cty).sum()})", fontsize=9, fontweight="bold")
    axes = sf.subplots(3, 1, sharex=True, gridspec_kw={"height_ratios": [len(order[b]) for b in BLOCKS]})
    for ax, blk in zip(axes, BLOCKS):
        for i, it in enumerate(order[blk]):
            x = d.loc[d.country == cty, it].dropna()
            pr = np.array([(x == k).mean() * 100 for k in range(1, 6)])
            left = -(pr[0] + pr[1] + pr[2] / 2)
            for k in range(5):
                ax.barh(i, pr[k], left=left, height=0.75, color=LIK_COLS[k], edgecolor="white", lw=0.3)
                left += pr[k]
        ax.axvline(0, color="black", lw=0.5)
        ax.set_yticks(range(len(order[blk])))
        if j == 0:
            q = T2.set_index("item").q_bh
            ax.set_yticklabels([f"{LABEL[it]} {star(q[it])}" for it in order[blk]], fontsize=6.4)
        else:
            ax.set_yticklabels([])
        ax.set_title(blk, loc="left", fontsize=7.5)
        ax.set_xlim(-100, 100); ax.set_xticks([-100, -50, 0, 50, 100])
    axes[-1].set_xticklabels(["100", "50", "0", "50", "100"])
    axes[-1].set_xlabel("% disagree  |  % agree")
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in LIK_COLS]
fig.legend(handles, LIK_LABS, loc="outside lower center", ncol=5)
savefig(fig, "Fig2_likert_diverging")

In [ ]:
# Figure S1: yes/no and tick-box questions with Wilson 95% CIs
t3s = T3.iloc[::-1].reset_index(drop=True)
fig, ax = plt.subplots(figsize=(7.2, 6.2), layout="constrained")
for cty, off, pre in [("Nigeria", 0.18, "NG"), ("Ghana", -0.18, "GH")]:
    m = t3s[f"{cty}_pct_yes"]
    ax.errorbar(m, t3s.index + off, xerr=[m - t3s[f"{pre}_lo"], t3s[f"{pre}_hi"] - m], fmt="o", ms=3.2,
                color=COL[cty], elinewidth=0.8, capsize=1.5, label=cty)
ax.set_yticks(t3s.index)
ax.set_yticklabels([f"{l} {star(q)}" for l, q in zip(t3s.label, t3s.q_bh)], fontsize=6.4)
ax.set_xlabel("Respondents answering yes or ticking the option (%), Wilson 95% CI")
ax.set_xlim(0, 100); ax.grid(axis="x", lw=0.3, alpha=0.5)
fig.legend(loc="outside lower center", ncol=2)
savefig(fig, "FigS1_binary_items")

## Step 3. Attitude segments and response styles: latent class analysis (R)

Several attitude measures are single five-point items, so we use **latent class analysis** (`poLCA`), which models answer categories directly. Seven indicators are cut into low, middle and high: harm perception, support for regulation and substitutes, shared responsibility, extra charges, government responsibility, making plastic cheaper, and engagement.

The rule for choosing the number of classes was fixed in advance. We take the lowest BIC among solutions where the smallest class holds at least 5% of respondents and entropy R² is at least 0.70. Country is **not** used to form the classes. It is compared afterwards.

The next cell writes the R script to `r_scripts/`, and the cell after it runs the script.

In [ ]:
%%writefile r_scripts/03_latent_classes.R
# Latent class analysis of policy orientations (pooled sample). Run from the notebook, with Rscript, or with source() in RStudio
# ---- find the project folder and load packages (details in r_scripts/00_setup.R; nothing to edit) ----
local({
  here <- c(gsub("~+~", " ", sub("^--file=", "", grep("^--file=", commandArgs(FALSE), value = TRUE)), fixed = TRUE),
            unlist(lapply(sys.frames(), function(f) f$ofile)),
            tryCatch(rstudioapi::getSourceEditorContext()$path, error = function(e) NULL))
  for (d in c(dirname(here[nzchar(here)]), getwd())) {
    d <- normalizePath(d, winslash = "/", mustWork = FALSE)
    for (k in 1:6) {
      f <- file.path(d, c("00_setup.R", "r_scripts/00_setup.R", "Plastic_NG_GH_project/r_scripts/00_setup.R"))
      if (any(file.exists(f))) { source(f[file.exists(f)][1], local = globalenv()); return(invisible()) }
      d <- dirname(d)
    }
  }
  stop("Cannot find r_scripts/00_setup.R. Run this script from inside the project folder (the one that contains data/processed/).")
})
use_packages(c("poLCA", "nnet"))
set.seed(20231101)
d <- read.csv(file.path(root, "data", "processed", "survey_merged_deidentified.csv"))
d <- d[d$flag_no_likert == 0, ]
harm <- c("flood", "erosion", "vector", "infiltration", "buildings", "toxic", "microbes", "nutrients",
          "bags_billions", "waterways_tons", "aquatic", "food", "terrestrial", "harmful_env")
d$HARM <- rowMeans(d[, harm], na.rm = TRUE); d$HARM[is.nan(d$HARM)] <- NA
d$SUPPORT <- rowMeans(d[, c("polluter_pays", "legal", "biodeg_alt", "cheap_bioplastic")], na.rm = TRUE)
d$SUPPORT[is.nan(d$SUPPORT)] <- NA
cut3c <- function(x) as.integer(cut(x, c(-Inf, 2.999, 3.999, Inf)))   # composite: <3, 3 to <4, >=4
cut3i <- function(x) as.integer(cut(x, c(-Inf, 2.5, 3.5, Inf)))       # single item: disagree, neutral, agree
L <- data.frame(HARM = cut3c(d$HARM), SUPPORT = cut3c(d$SUPPORT), everybody = cut3i(d$everybody),
                charges = cut3i(d$charges), gov_resp = cut3i(d$gov_resp), cheaper_plastic = cut3i(d$cheaper_plastic),
                engagement = as.integer(cut(d$engagement, c(-Inf, 1.01, 2.01, Inf))))
f <- cbind(HARM, SUPPORT, everybody, charges, gov_resp, cheaper_plastic, engagement) ~ 1
entropy_r2 <- function(m) { p <- m$posterior; k <- ncol(p); if (k == 1) return(NA)
  1 - sum(-p * log(pmax(p, 1e-12))) / (nrow(p) * log(k)) }

# fit 1 to 7 classes, 50 random starts each
fits <- list(); rows <- list()
for (k in 1:7) {
  m <- poLCA(f, L, nclass = k, nrep = 50, maxiter = 5000, na.rm = FALSE, verbose = FALSE)
  fits[[k]] <- m; modal <- apply(m$posterior, 1, which.max)
  rows[[k]] <- data.frame(k = k, loglik = m$llik, npar = m$npar, BIC = m$bic, AIC = m$aic,
                          aBIC = -2 * m$llik + m$npar * log((m$N + 2) / 24), entropy_R2 = entropy_r2(m),
                          smallest_class_pct = 100 * min(tabulate(modal, k)) / length(modal),
                          avg_post_min = if (k == 1) 1 else min(sapply(1:k, function(j) mean(m$posterior[modal == j, j]))),
                          Gsq = m$Gsq, resid_df = m$resid.df)
}
fs <- do.call(rbind, rows); write.csv(round(fs, 3), file.path(tab, "T10a_lca_model_selection.csv"), row.names = FALSE)
ok <- fs[fs$smallest_class_pct >= 5 & (fs$entropy_R2 >= 0.70 | fs$k == 1), ]
K <- ok$k[which.min(ok$BIC)]; m <- fits[[K]]
ord <- order(-sapply(1:K, function(j) m$probs$SUPPORT[j, 3]))   # class 1 = highest support
post <- m$posterior[, ord, drop = FALSE]
d$class <- apply(post, 1, which.max); d$class_post <- apply(post, 1, max)

ip <- do.call(rbind, lapply(names(m$probs), function(v) { pr <- m$probs[[v]][ord, , drop = FALSE]
  data.frame(indicator = v, class = 1:K, p_low = round(pr[, 1], 3), p_mid = round(pr[, 2], 3), p_high = round(pr[, 3], 3)) }))
write.csv(ip, file.path(tab, "T10b_lca_item_probabilities.csv"), row.names = FALSE)

ct <- table(d$class, d$country)[, c("Nigeria", "Ghana")]; cs <- suppressWarnings(chisq.test(ct))
pw <- sapply(c("Nigeria", "Ghana"), function(c) colSums(post[d$country == c, , drop = FALSE]) / sum(d$country == c) * 100)
comp <- data.frame(class = 1:K, Nigeria_n = as.integer(ct[, 1]), Ghana_n = as.integer(ct[, 2]),
                   Nigeria_pct = round(100 * ct[, 1] / sum(ct[, 1]), 1), Ghana_pct = round(100 * ct[, 2] / sum(ct[, 2]), 1),
                   Nigeria_pct_postweighted = round(pw[, 1], 1), Ghana_pct_postweighted = round(pw[, 2], 1),
                   class_size_pct = round(100 * colMeans(post), 1), mean_posterior = round(tapply(d$class_post, d$class, mean), 3),
                   chi2 = round(cs$statistic, 2), df = cs$parameter, p = signif(cs$p.value, 3),
                   cramers_v = round(sqrt(cs$statistic / (sum(ct) * (min(dim(ct)) - 1))), 3), selected_k = K)
write.csv(comp, file.path(tab, "T10c_lca_country_composition.csv"), row.names = FALSE)

# membership adjusted for demographics (multinomial logit; reference = largest class)
d$ghana <- as.numeric(d$country == "Ghana"); d$female <- ifelse(is.na(d$gender), NA, as.numeric(d$gender == "Female"))
d$age_o <- as.numeric(factor(d$age, levels = c("18-27", "28-37", "38-47", "48+")))
d$city <- ifelse(is.na(d$community), NA, as.numeric(d$community == "City"))
d$student <- ifelse(is.na(d$occ_group), NA, as.numeric(d$occ_group == "Student"))
ref <- which.max(table(d$class)); d$class_f <- relevel(factor(d$class), ref = as.character(ref))
mn <- multinom(class_f ~ ghana + female + age_o + city + income_top + student, data = d, trace = FALSE)
co <- summary(mn)$coefficients; se <- summary(mn)$standard.errors
if (is.null(dim(co))) { co <- t(as.matrix(co)); se <- t(as.matrix(se)); rownames(co) <- rownames(se) <- levels(d$class_f)[2] }
mt <- do.call(rbind, lapply(rownames(co), function(r) data.frame(contrast = paste0("C", r, " vs C", ref), term = colnames(co),
  b = round(co[r, ], 3), rrr = round(exp(co[r, ]), 3), se = round(se[r, ], 3), p = signif(2 * pnorm(-abs(co[r, ] / se[r, ])), 3), row.names = NULL)))
write.csv(mt, file.path(tab, "T10d_lca_multinomial_membership.csv"), row.names = FALSE)
cm <- aggregate(d[, c("HARM", "SUPPORT", "everybody", "charges", "gov_resp", "cheaper_plastic", "engagement")],
                list(class = d$class), function(x) round(mean(x, na.rm = TRUE), 2))
write.csv(cm, file.path(tab, "T10e_lca_class_means.csv"), row.names = FALSE)
write.csv(d[, c("resp_id", "country", "class", "class_post")], file.path(tab, "T10f_lca_assignments.csv"), row.names = FALSE)
print(fs); print(comp); print(cm)

In [ ]:
run_r("03_latent_classes.R")
pd.read_csv(TAB / "T10c_lca_country_composition.csv")

In [ ]:
# Country gaps overall, without the uniform-disagreement class, and within classes (Table T10h),
# plus regressions adjusted for demographics, engagement and harm perception (Table T10i)
cls = pd.read_csv(TAB / "T10f_lca_assignments.csv")[["resp_id", "class"]]
cmeans = pd.read_csv(TAB / "T10e_lca_class_means.csv")
UNIFORM = int(cmeans.sort_values("SUPPORT")["class"].iloc[0])            # lowest-support class
TOP = int(cmeans.sort_values("SUPPORT", ascending=False)["class"].iloc[0])
MID = [c for c in cmeans["class"] if c not in (UNIFORM, TOP)][0]
CLASS_NAME = {TOP: "Consistent supporters", MID: "Moderate supporters", UNIFORM: "Uniform disagreement"}
x = d.merge(cls, on="resp_id", how="left")
x["HARM"] = x[HARM].mean(axis=1); x["SUPPORT"] = x[SUPP].mean(axis=1)
OUT = {"HARM": "Harm perception (mean of 14)", "SUPPORT": "Regulation and substitutes (mean of 4)",
       "everybody": "Shared responsibility", "charges": "Extra charges", "gov_resp": "Government responsibility",
       "cheaper_plastic": "Make plastic cheaper"}
subsets = {"All respondents": x, "Excluding uniform-disagreement class": x[x["class"] != UNIFORM],
           "Within consistent supporters": x[x["class"] == TOP], "Within moderate supporters": x[x["class"] == MID]}
rows = []
for sname, s in subsets.items():
    for v, lab in OUT.items():
        a, b = s.loc[s.country == "Nigeria", v].dropna(), s.loc[s.country == "Ghana", v].dropna()
        u, p = stats.mannwhitneyu(b, a)
        rows.append({"subset": sname, "outcome": lab, "Nigeria_n": len(a), "Nigeria_mean": a.mean(), "Ghana_n": len(b),
                     "Ghana_mean": b.mean(), "r_rb_GH_minus_NG": 2 * u / (len(a) * len(b)) - 1, "p": p})
T10h = pd.DataFrame(rows)
T10h["q_bh_within_subset"] = T10h.groupby("subset").p.transform(lambda q: multipletests(q, method="fdr_bh")[1])
T10h.round(4).to_csv(TAB / "T10h_country_gaps_by_class.csv", index=False)

x["female"] = (x.gender == "Female").astype(float).where(x.gender.notna())
x["city"] = (x.community == "City").astype(float).where(x.community.notna())
x["student"] = (x.occ_group == "Student").astype(float).where(x.occ_group.notna())
x["ghana"] = (x.country == "Ghana").astype(float)
rows = []
for sname, s in [("All respondents", x), ("Excluding uniform-disagreement class", x[x["class"] != UNIFORM])]:
    for v in OUT:
        rhs = "ghana + C(age) + female + city + income_top + student + engagement" + ("" if v == "HARM" else " + HARM")
        m = smf.ols(f"{v} ~ {rhs}", data=s).fit(cov_type="HC3")
        ci = m.conf_int().loc["ghana"]
        rows.append({"subset": sname, "outcome": OUT[v], "b_ghana": m.params["ghana"], "se": m.bse["ghana"],
                     "p": m.pvalues["ghana"], "ci_lo": ci[0], "ci_hi": ci[1], "n": int(m.nobs), "controls_harm": v != "HARM"})
T10i = pd.DataFrame(rows); T10i.round(4).to_csv(TAB / "T10i_adjusted_country_effects.csv", index=False)
T10h.pivot(index="outcome", columns="subset", values="r_rb_GH_minus_NG").round(2)

### Is the uniform-disagreement class a response style? Checks that do not depend on the class definition

Removing the class shrinks the country gap partly by construction, because the class is defined by low harm and support scores. Two checks avoid that problem:

1. **A reversed item.** "Throwing plastic waste in the trash will not harm the environment" is worded in the opposite direction to the harm items and is **not** one of the class indicators. People who truly believe plastic is harmless should agree with it. People who disagree with every statement will disagree with it too.
2. **Content-free response-style indices** over all 25 Likert items (Meade and Craig, 2012): the spread of a person's answers (individual SD), the share of extreme answers (1 or 5), the share of midpoint answers (3) and the longest run of identical answers. We then ask how much of the country gap in harm and support remains once these indices are held constant (Table T10l).

In [ ]:
L25 = d[LIKERT]
rs = d[["resp_id", "country", "trash_no_harm", "waste_wealth", "pref_recycle"]].copy()
rs["indiv_sd"] = L25.std(axis=1, ddof=0).where(L25.notna().sum(axis=1) >= 10)
rs["extreme_share"] = ((L25 == 1) | (L25 == 5)).sum(axis=1) / L25.notna().sum(axis=1)
rs["midpoint_share"] = (L25 == 3).sum(axis=1) / L25.notna().sum(axis=1)

def longest_run(row):
    v = [a for a in row if not np.isnan(a)]
    best = cur = 1 if v else 0
    for i in range(1, len(v)):
        cur = cur + 1 if v[i] == v[i - 1] else 1
        best = max(best, cur)
    return best

rs["longest_run"] = L25.apply(lambda r: longest_run(r.values.astype(float)), axis=1)
rs["HARM"], rs["SUPPORT"] = d[HARM].mean(axis=1), d[SUPP].mean(axis=1)
rs = rs[d.flag_no_likert == 0].merge(cls, on="resp_id", how="left")

# (1) the reversed item and two non-grid items, by class
by_class = rs.groupby("class").agg(n=("resp_id", "size"), reversed_item_mean=("trash_no_harm", "mean"),
                                   reversed_item_disagree_pct=("trash_no_harm", lambda v: 100 * (v.dropna() <= 2).mean()),
                                   sees_wealth_yes_pct=("waste_wealth", lambda v: 100 * (v.dropna() == 1).mean()),
                                   sees_wealth_yes_or_maybe_pct=("waste_wealth", lambda v: 100 * (v.dropna() >= 0.5).mean()),
                                   prefers_recycling_pct=("pref_recycle", lambda v: 100 * v.dropna().mean()),
                                   indiv_sd=("indiv_sd", "mean"), extreme_share=("extreme_share", "mean"),
                                   midpoint_share=("midpoint_share", "mean"), longest_run=("longest_run", "mean")).reset_index()
by_class["class_name"] = by_class["class"].map(CLASS_NAME)
by_class.round(3).to_csv(TAB / "T10m_response_style_by_class.csv", index=False)

# (2) country gap (in SD units) before and after holding the response-style indices constant
rs["ghana"] = (rs.country == "Ghana").astype(int)
STYLE = "indiv_sd + extreme_share + midpoint_share + longest_run"
rows = []
for v, lab in [("HARM", "Harm perception"), ("SUPPORT", "Regulation and substitutes")]:
    z = rs.dropna(subset=[v, "indiv_sd"]).copy(); z["y"] = (z[v] - z[v].mean()) / z[v].std()
    for spec, rhs in [("Unadjusted", "ghana"), ("Adjusted for response-style indices", "ghana + " + STYLE)]:
        m = smf.ols(f"y ~ {rhs}", data=z).fit(cov_type="HC3"); ci = m.conf_int().loc["ghana"]
        rows.append({"outcome": lab, "model": spec, "gap_sd_ghana_minus_nigeria": m.params["ghana"], "ci_lo": ci[0],
                     "ci_hi": ci[1], "p": m.pvalues["ghana"], "n": int(m.nobs)})
T10l = pd.DataFrame(rows)
T10l["share_of_gap_removed_pct"] = 100 * (1 - T10l.gap_sd_ghana_minus_nigeria / T10l.groupby("outcome").gap_sd_ghana_minus_nigeria.transform("first"))
T10l.round(4).to_csv(TAB / "T10l_gap_adjusted_for_response_style.csv", index=False)
style_by_country = rs.groupby("country")[["indiv_sd", "extreme_share", "midpoint_share", "longest_run"]].mean().round(3)
print(style_by_country)
display(by_class.round(2)); T10l.round(3)

### Are the latent classes the same in both countries?

A latent class model is only comparable across countries if the classes mean the same thing in each. Following Kankaraš et al. (2018), we compare a **homogeneous** model (one set of item-response probabilities for both countries, class sizes free by country) with a **heterogeneous** model (a separate three-class model in each country). A lower BIC for the homogeneous model supports comparing class shares across countries.

In [ ]:
%%writefile r_scripts/05_lca_invariance.R
# Are the three latent classes the same in both countries? Run from the notebook, with Rscript, or with source() in RStudio
# Compares a homogeneous model (same item-response probabilities in both countries, class sizes free by country)
# with a heterogeneous model (a separate 3-class model in each country), following Kankaras et al. (2018).
# ---- find the project folder and load packages (details in r_scripts/00_setup.R; nothing to edit) ----
local({
  here <- c(gsub("~+~", " ", sub("^--file=", "", grep("^--file=", commandArgs(FALSE), value = TRUE)), fixed = TRUE),
            unlist(lapply(sys.frames(), function(f) f$ofile)),
            tryCatch(rstudioapi::getSourceEditorContext()$path, error = function(e) NULL))
  for (d in c(dirname(here[nzchar(here)]), getwd())) {
    d <- normalizePath(d, winslash = "/", mustWork = FALSE)
    for (k in 1:6) {
      f <- file.path(d, c("00_setup.R", "r_scripts/00_setup.R", "Plastic_NG_GH_project/r_scripts/00_setup.R"))
      if (any(file.exists(f))) { source(f[file.exists(f)][1], local = globalenv()); return(invisible()) }
      d <- dirname(d)
    }
  }
  stop("Cannot find r_scripts/00_setup.R. Run this script from inside the project folder (the one that contains data/processed/).")
})
use_packages("poLCA")
set.seed(20231101)
d <- read.csv(file.path(root, "data", "processed", "survey_merged_deidentified.csv"))
d <- d[d$flag_no_likert == 0, ]
harm <- c("flood", "erosion", "vector", "infiltration", "buildings", "toxic", "microbes", "nutrients",
          "bags_billions", "waterways_tons", "aquatic", "food", "terrestrial", "harmful_env")
d$HARM <- rowMeans(d[, harm], na.rm = TRUE); d$HARM[is.nan(d$HARM)] <- NA
d$SUPPORT <- rowMeans(d[, c("polluter_pays", "legal", "biodeg_alt", "cheap_bioplastic")], na.rm = TRUE)
d$SUPPORT[is.nan(d$SUPPORT)] <- NA
cut3c <- function(x) as.integer(cut(x, c(-Inf, 2.999, 3.999, Inf)))
cut3i <- function(x) as.integer(cut(x, c(-Inf, 2.5, 3.5, Inf)))
L <- data.frame(HARM = cut3c(d$HARM), SUPPORT = cut3c(d$SUPPORT), everybody = cut3i(d$everybody),
                charges = cut3i(d$charges), gov_resp = cut3i(d$gov_resp), cheaper_plastic = cut3i(d$cheaper_plastic),
                engagement = as.integer(cut(d$engagement, c(-Inf, 1.01, 2.01, Inf))), ghana = as.numeric(d$country == "Ghana"))
f0 <- cbind(HARM, SUPPORT, everybody, charges, gov_resp, cheaper_plastic, engagement) ~ 1
fit <- function(f, dat) poLCA(f, dat, nclass = 3, nrep = 50, maxiter = 5000, na.rm = FALSE, verbose = FALSE)

hom <- fit(cbind(HARM, SUPPORT, everybody, charges, gov_resp, cheaper_plastic, engagement) ~ ghana, L)
ng <- fit(f0, L[L$ghana == 0, ]); gh <- fit(f0, L[L$ghana == 1, ])
N <- nrow(L)
ll_het <- ng$llik + gh$llik; np_het <- ng$npar + gh$npar
res <- data.frame(model = c("Homogeneous (equal item probabilities, class sizes by country)", "Heterogeneous (separate model per country)"),
                  loglik = c(hom$llik, ll_het), npar = c(hom$npar, np_het))
res$BIC <- -2 * res$loglik + res$npar * log(N); res$AIC <- -2 * res$loglik + 2 * res$npar
res$LR <- c(NA, 2 * (ll_het - hom$llik)); res$df <- c(NA, np_het - hom$npar)
res$p <- c(NA, pchisq(res$LR[2], res$df[2], lower.tail = FALSE))
write.csv(res, file.path(tab, "T10j_lca_invariance.csv"), row.names = FALSE)

# Class profiles per country, ordered like the pooled model (class 1 = highest support)
prof <- function(m, lab) { ord <- order(-sapply(1:3, function(j) m$probs$SUPPORT[j, 3]))
  do.call(rbind, lapply(names(m$probs), function(v) data.frame(model = lab, indicator = v, class = 1:3,
    p_high = round(m$probs[[v]][ord, 3], 3), p_low = round(m$probs[[v]][ord, 1], 3), share = round(m$P[ord], 3)))) }
pr <- rbind(prof(ng, "Nigeria only"), prof(gh, "Ghana only"), prof(hom, "Pooled homogeneous"))
write.csv(pr, file.path(tab, "T10k_lca_profiles_by_country.csv"), row.names = FALSE)
print(res); print(pr[pr$indicator %in% c("HARM", "SUPPORT", "cheaper_plastic"), ])

In [ ]:
run_r("05_lca_invariance.R")
pd.read_csv(TAB / "T10j_lca_invariance.csv").round(3)

## Step 4. Measurement, invariance and the structural model (R, lavaan)

This is the longest step: about 30 to 40 minutes, mostly for the sensitivity checks. The R script:

1. runs a polychoric EFA, then a two-factor CFA (harm perception: 14 items; support for regulation and substitutes: 4 items) with robust ML and full-information ML for missing answers;
2. reports reliability (alpha, omega), AVE and HTMT;
3. tests **measurement invariance** across countries (configural, metric, scalar, strict), plus an ordinal WLSMV check, and compares latent means;
4. fits the **multi-group SEM**: engagement → harm perception → shared-responsibility norm → support, plus three single-item instrument stances. It tests whether each path differs between countries and computes Monte Carlo CIs for indirect effects;
5. runs seven sensitivity checks: quality-flagged responses removed, a random-intercept response-style factor, an ordinal estimator, an alternative specification, fixed covariates, and the uniform-disagreement class removed.

lavaan may warn that the parameter covariance matrix "does not appear to be positive definite". The near-zero eigenvalues match the equality constraints between the groups (and the binary covariates), so the warning is expected and harmless.

In [ ]:
%%writefile r_scripts/04_measurement_sem.R
# Measurement model, invariance and multi-group SEM. Run from the notebook, with Rscript, or with source() in RStudio
#
# Inputs : data/processed/survey_merged_deidentified.csv
# Outputs: outputs/tables/T4_* to T9_*, outputs/models/*.txt
#
# Main estimator: robust maximum likelihood (MLR) with full-information ML for missing data,
# treating 5-point items as continuous (Rhemtulla et al., 2012). Ordinal invariance (WLSMV,
# Wu & Estabrook, 2016) and an ordinal structural model are run as robustness checks.
# ---- find the project folder and load packages (details in r_scripts/00_setup.R; nothing to edit) ----
local({
  here <- c(gsub("~+~", " ", sub("^--file=", "", grep("^--file=", commandArgs(FALSE), value = TRUE)), fixed = TRUE),
            unlist(lapply(sys.frames(), function(f) f$ofile)),
            tryCatch(rstudioapi::getSourceEditorContext()$path, error = function(e) NULL))
  for (d in c(dirname(here[nzchar(here)]), getwd())) {
    d <- normalizePath(d, winslash = "/", mustWork = FALSE)
    for (k in 1:6) {
      f <- file.path(d, c("00_setup.R", "r_scripts/00_setup.R", "Plastic_NG_GH_project/r_scripts/00_setup.R"))
      if (any(file.exists(f))) { source(f[file.exists(f)][1], local = globalenv()); return(invisible()) }
      d <- dirname(d)
    }
  }
  stop("Cannot find r_scripts/00_setup.R. Run this script from inside the project folder (the one that contains data/processed/).")
})
use_packages(c("lavaan", "semTools", "psych", "GPArotation"))
set.seed(20231101)
d <- read.csv(file.path(root, "data", "processed", "survey_merged_deidentified.csv"))
d$country <- factor(d$country, levels = c("Nigeria", "Ghana"))
d$female  <- ifelse(is.na(d$gender), NA, as.numeric(d$gender == "Female"))
d$age_o   <- as.numeric(factor(d$age, levels = c("18-27", "28-37", "38-47", "48+")))
d$city    <- ifelse(is.na(d$community), NA, as.numeric(d$community == "City"))
d$student <- ifelse(is.na(d$occ_group), NA, as.numeric(d$occ_group == "Student"))
d$ghana   <- as.numeric(d$country == "Ghana")

harm <- c("flood", "erosion", "vector", "infiltration", "buildings", "toxic", "microbes", "nutrients",
          "bags_billions", "waterways_tons", "aquatic", "food", "terrestrial", "harmful_env")
supp <- c("polluter_pays", "legal", "biodeg_alt", "cheap_bioplastic")
sink_txt <- function(obj, f) { capture.output(obj, file = file.path(mod, f)) }
fitm <- c("chisq.scaled", "df.scaled", "pvalue.scaled", "cfi.robust", "tli.robust",
          "rmsea.robust", "rmsea.ci.lower.robust", "rmsea.ci.upper.robust", "srmr")
getfit <- function(f, label) {
  x <- fitMeasures(f, fitm)
  if (is.na(x["cfi.robust"])) {  # WLSMV with missing data: robust indices unavailable, use scaled
    y <- fitMeasures(f, c("cfi.scaled", "tli.scaled", "rmsea.scaled", "rmsea.ci.lower.scaled", "rmsea.ci.upper.scaled"))
    x[c("cfi.robust", "tli.robust", "rmsea.robust", "rmsea.ci.lower.robust", "rmsea.ci.upper.robust")] <- y
  }
  data.frame(model = label, as.list(round(unclass(x), 4)), row.names = NULL, check.names = FALSE)
}

# ------------------------------------------------------------------ 1. EFA (polychoric, pooled)
efa_items <- c(harm, "charges", "gov_resp", "everybody", supp, "cheaper_plastic")
pc <- polychoric(d[, efa_items], correct = 0)$rho
pa <- fa.parallel(pc, n.obs = sum(complete.cases(d[, efa_items])), fa = "fa", plot = FALSE)
efa <- fa(pc, nfactors = 3, n.obs = sum(complete.cases(d[, efa_items])), rotate = "oblimin", fm = "wls")
L <- round(unclass(efa$loadings), 3)
write.csv(data.frame(item = rownames(L), L, communality = round(efa$communality, 3)),
          file.path(tab, "T4a_efa_polychoric_loadings.csv"), row.names = FALSE)
sink_txt(list(parallel_nfact = pa$nfact, efa = print(efa, cut = 0.3, sort = TRUE)), "efa.txt")

# ------------------------------------------------------------------ 2. CFA measurement model
# Correlated residuals are restricted to item pairs with near-identical wording (stated a priori):
# "billions of bags discarded" ~ "millions of tonnes reach waterways" (volume statements),
# "harms aquatic animals" ~ "harms terrestrial animals" (parallel stems),
# "biodegradable alternatives" ~ "affordable bioplastics" (substitution items).
cfa_m <- '
HARM =~ flood + erosion + vector + infiltration + buildings + toxic + microbes + nutrients +
        bags_billions + waterways_tons + aquatic + food + terrestrial + harmful_env
SUPPORT =~ polluter_pays + legal + biodeg_alt + cheap_bioplastic
bags_billions ~~ waterways_tons
aquatic ~~ terrestrial
biodeg_alt ~~ cheap_bioplastic
'
cfa_m0 <- gsub("bags_billions ~~ waterways_tons\naquatic ~~ terrestrial\nbiodeg_alt ~~ cheap_bioplastic\n", "", cfa_m)
fit_list <- list()
for (cty in c("Pooled", "Nigeria", "Ghana")) {
  dd <- if (cty == "Pooled") d else d[d$country == cty, ]
  f0 <- cfa(cfa_m0, dd, estimator = "MLR", missing = "fiml")
  f1 <- cfa(cfa_m, dd, estimator = "MLR", missing = "fiml")
  fit_list[[cty]] <- f1
  write.csv(rbind(getfit(f0, paste(cty, "no residual covariances")), getfit(f1, paste(cty, "final"))),
            file.path(tab, paste0("T4b_cfa_fit_", cty, ".csv")), row.names = FALSE)
  sink_txt(summary(f1, fit.measures = TRUE, standardized = TRUE), paste0("cfa_", cty, ".txt"))
}
cfa_fit <- do.call(rbind, lapply(c("Pooled", "Nigeria", "Ghana"), function(c)
  read.csv(file.path(tab, paste0("T4b_cfa_fit_", c, ".csv")))))
write.csv(cfa_fit, file.path(tab, "T4b_cfa_fit.csv"), row.names = FALSE)
for (c in c("Pooled", "Nigeria", "Ghana")) file.remove(file.path(tab, paste0("T4b_cfa_fit_", c, ".csv")))

# loadings, omega, AVE, HTMT per country
rel_rows <- list(); load_rows <- list()
for (cty in c("Nigeria", "Ghana")) {
  f <- fit_list[[cty]]
  st <- standardizedSolution(f); st <- st[st$op == "=~", ]
  load_rows[[cty]] <- data.frame(country = cty, factor = st$lhs, item = st$rhs,
                                 std_loading = round(st$est.std, 3), se = round(st$se, 3))
  om <- compRelSEM(f); av <- AVE(f)
  dd <- d[d$country == cty, ]
  ht <- htmt(cfa_m0, data = dd[, c(harm, supp)], missing = "pairwise")
  alpha_h <- psych::alpha(dd[, harm])$total$raw_alpha
  alpha_s <- psych::alpha(dd[, supp])$total$raw_alpha
  lv <- lavInspect(f, "cor.lv")
  rel_rows[[cty]] <- data.frame(country = cty, construct = c("HARM", "SUPPORT"),
                                alpha = round(c(alpha_h, alpha_s), 3),
                                omega = round(as.numeric(om[c("HARM", "SUPPORT")]), 3),
                                AVE = round(as.numeric(av[c("HARM", "SUPPORT")]), 3),
                                sqrt_AVE = round(sqrt(as.numeric(av[c("HARM", "SUPPORT")])), 3),
                                latent_r = round(lv["HARM", "SUPPORT"], 3),
                                HTMT = round(ht["SUPPORT", "HARM"], 3))
}
write.csv(do.call(rbind, load_rows), file.path(tab, "T4c_cfa_loadings.csv"), row.names = FALSE)
write.csv(do.call(rbind, rel_rows), file.path(tab, "T4d_reliability_validity.csv"), row.names = FALSE)

# ------------------------------------------------------------------ 3. Measurement invariance (MLR, continuous)
inv_fit <- function(eq, partial = NULL) cfa(cfa_m, d, group = "country", estimator = "MLR",
                                            missing = "fiml", group.equal = eq, group.partial = partial)
m_conf <- inv_fit(character(0))
m_metr <- inv_fit("loadings")
m_scal <- inv_fit(c("loadings", "intercepts"))
m_strc <- inv_fit(c("loadings", "intercepts", "residuals"))
# partial scalar invariance: free intercepts flagged by the score test, one at a time, until dCFI <= .010
score <- lavTestScore(m_scal)$uni
pt <- parTable(m_scal)
score$lhs_lab <- pt$label[match(score$lhs, pt$plabel)]
score$rhs_lab <- pt$label[match(score$rhs, pt$plabel)]
freed <- character(0); m_ps <- m_scal
cfi_metr <- fitMeasures(m_metr, "cfi.robust")
sc <- score[order(-score$X2), ]
for (i in seq_len(nrow(sc))) {
  if (cfi_metr - fitMeasures(m_ps, "cfi.robust") <= 0.010) break
  prow <- pt[pt$plabel == sc$lhs[i], ]
  if (nrow(prow) == 0 || prow$op != "~1") next
  term <- paste0(prow$lhs, "~1")
  if (term %in% freed) next
  freed <- c(freed, term)
  m_ps <- inv_fit(c("loadings", "intercepts"), freed)
}
inv_tab <- rbind(getfit(m_conf, "Configural"), getfit(m_metr, "Metric (loadings)"),
                 getfit(m_scal, "Scalar (loadings + intercepts)"),
                 getfit(m_ps, paste0("Partial scalar (free: ", paste(freed, collapse = ", "), ")")),
                 getfit(m_strc, "Strict (+ residual variances)"))
inv_tab$delta_cfi <- c(NA, diff(inv_tab$cfi.robust))
inv_tab$delta_rmsea <- c(NA, diff(inv_tab$rmsea.robust))
inv_tab$delta_srmr <- c(NA, diff(inv_tab$srmr))
# scaled chi-square difference tests vs the preceding nested model
lrt <- function(a, b) { x <- lavTestLRT(a, b, method = "satorra.bentler.2001")
  c(chisq_diff = x[2, "Chisq diff"], df_diff = x[2, "Df diff"], p_diff = x[2, "Pr(>Chisq)"]) }
dt <- rbind(c(NA, NA, NA), lrt(m_conf, m_metr), lrt(m_metr, m_scal), lrt(m_metr, m_ps), lrt(m_scal, m_strc))
inv_tab <- cbind(inv_tab, round(dt, 4))
inv_tab$comparison <- c("", "vs configural", "vs metric", "vs metric", "vs scalar")
if (length(freed) == 0) inv_tab <- inv_tab[-4, ]  # full scalar invariance held; no partial model needed
write.csv(inv_tab, file.path(tab, "T5a_invariance_mlr.csv"), row.names = FALSE)

# latent mean differences under (partial) scalar invariance; Nigeria is the reference (mean 0)
lm_ps <- parameterEstimates(m_ps)
lat <- lm_ps[lm_ps$op == "~1" & lm_ps$lhs %in% c("HARM", "SUPPORT") & lm_ps$group == 2, ]
lv_var <- lm_ps[lm_ps$op == "~~" & lm_ps$lhs == lm_ps$rhs & lm_ps$lhs %in% c("HARM", "SUPPORT"), ]
pooled_sd <- sapply(c("HARM", "SUPPORT"), function(f) sqrt(mean(lv_var$est[lv_var$lhs == f])))
lat$d <- lat$est / pooled_sd[lat$lhs]
write.csv(data.frame(factor = lat$lhs, ghana_minus_nigeria = round(lat$est, 3), se = round(lat$se, 3),
                     z = round(lat$z, 2), p = signif(lat$pvalue, 3), ci_lo = round(lat$ci.lower, 3),
                     ci_hi = round(lat$ci.upper, 3), cohen_d = round(lat$d, 3)),
          file.path(tab, "T5b_latent_mean_differences.csv"), row.names = FALSE)

# ------------------------------------------------------------------ 3b. Ordinal invariance (WLSMV) robustness
dord <- d
for (v in c(harm, supp)) dord[[v]] <- ordered(dord[[v]], levels = 1:5)
# collapse categories 1-2 where any group has an empty category (needed for threshold invariance)
for (v in c(harm, supp)) {
  tb <- table(dord[[v]], dord$country)
  if (any(tb == 0)) dord[[v]] <- ordered(pmax(as.numeric(dord[[v]]), 2), levels = 2:5)
}
ord_rows <- list(); prev <- NULL
for (lvl in c("configural", "thresholds", "loadings", "intercepts")) {
  syn <- measEq.syntax(configural.model = cfa_m, data = dord, ordered = c(harm, supp),
                       parameterization = "theta", ID.fac = "std.lv", ID.cat = "Wu.Estabrook.2016",
                       group = "country",
                       group.equal = switch(lvl, configural = "", thresholds = "thresholds",
                                            loadings = c("thresholds", "loadings"),
                                            intercepts = c("thresholds", "loadings", "intercepts")))
  f <- cfa(as.character(syn), dord, group = "country", ordered = c(harm, supp),
           estimator = "WLSMV", parameterization = "theta")
  ord_rows[[lvl]] <- getfit(f, paste("Ordinal", lvl))
  ord_rows[[lvl]]$chisq_diff_p <- if (is.null(prev)) NA else
    round(lavTestLRT(prev, f, method = "satorra.2000")[2, "Pr(>Chisq)"], 4)
  prev <- f
}
ordt <- do.call(rbind, ord_rows)
ordt$delta_cfi <- c(NA, diff(ordt$cfi.robust)); ordt$delta_rmsea <- c(NA, diff(ordt$rmsea.robust))
write.csv(ordt, file.path(tab, "T5c_invariance_wlsmv_ordinal.csv"), row.names = FALSE)

# ------------------------------------------------------------------ 4. Multi-group structural model
# Value-Belief-Norm chain: engagement -> harm perception (awareness of consequences) ->
# shared-responsibility norm ("everybody can play a role") -> policy support.
# Three single-item outcomes capture other instrument stances: price charges, government
# responsibility, and affordability-first ("make plastic cheaper").
paths <- function(g) {
  s <- function(x) paste0("c(", paste0(x, "_NG"), ",", paste0(x, "_GH"), ")*")
  paste0('
HARM =~ flood + erosion + vector + infiltration + buildings + toxic + microbes + nutrients +
        bags_billions + waterways_tons + aquatic + food + terrestrial + harmful_env
SUPPORT =~ polluter_pays + legal + biodeg_alt + cheap_bioplastic
bags_billions ~~ waterways_tons
aquatic ~~ terrestrial
biodeg_alt ~~ cheap_bioplastic
HARM ~ ', s("a1"), 'engagement + age_o + female + city + income_top + student
everybody ~ ', s("b1"), 'HARM + ', s("b2"), 'engagement + age_o + female + city + income_top + student
SUPPORT ~ ', s("c1"), 'everybody + ', s("c2"), 'HARM + ', s("c3"), 'engagement + age_o + female + city + income_top + student
charges ~ ', s("d1"), 'everybody + ', s("d2"), 'HARM + ', s("d3"), 'engagement
gov_resp ~ ', s("e1"), 'everybody + ', s("e2"), 'HARM + ', s("e3"), 'engagement
cheaper_plastic ~ ', s("f1"), 'everybody + ', s("f2"), 'HARM + ', s("f3"), 'engagement
charges ~~ gov_resp + cheaper_plastic
gov_resp ~~ cheaper_plastic
SUPPORT ~~ charges + gov_resp + cheaper_plastic
everybody ~~ 0*charges + 0*gov_resp + 0*cheaper_plastic
')
}
ind <- '
ind_HARM_NORM_SUP_NG := a1_NG*b1_NG*c1_NG
ind_HARM_NORM_SUP_GH := a1_GH*b1_GH*c1_GH
ind_ENG_SUP_total_NG := c3_NG + a1_NG*c2_NG + b2_NG*c1_NG + a1_NG*b1_NG*c1_NG
ind_ENG_SUP_total_GH := c3_GH + a1_GH*c2_GH + b2_GH*c1_GH + a1_GH*b1_GH*c1_GH
HARM_to_SUP_via_NORM_NG := b1_NG*c1_NG
HARM_to_SUP_via_NORM_GH := b1_GH*c1_GH
HARM_to_SUP_total_NG := c2_NG + b1_NG*c1_NG
HARM_to_SUP_total_GH := c2_GH + b1_GH*c1_GH
HARM_to_charges_via_NORM_NG := b1_NG*d1_NG
HARM_to_charges_via_NORM_GH := b1_GH*d1_GH
diff_HARM_to_SUP_via_NORM := HARM_to_SUP_via_NORM_GH - HARM_to_SUP_via_NORM_NG
diff_ind_ENG_SUP_total := ind_ENG_SUP_total_GH - ind_ENG_SUP_total_NG
'
sem_syn <- paste(paths(), ind)
eq_meas <- c("loadings", "intercepts")
# fixed.x = FALSE keeps respondents with missing covariates in the likelihood. lavaan then warns that the
# parameter covariance matrix is not positive definite: the near-zero eigenvalues correspond exactly to the
# between-group equality constraints plus the means and variances of binary covariates (checked by
# eigen-decomposition), so the warning is harmless. Robustness row "Covariates fixed" uses fixed.x = TRUE.
fit_sem <- function(dat, syn = sem_syn, partial = freed, fx = FALSE, ...) sem(syn, dat, group = "country",
  estimator = "MLR", missing = "fiml", fixed.x = fx, group.equal = eq_meas, group.partial = partial, ...)
s_free <- fit_sem(d)
sink_txt(summary(s_free, fit.measures = TRUE, standardized = TRUE, rsquare = TRUE), "sem_multigroup.txt")

pe <- parameterEstimates(s_free, standardized = TRUE)
pe$country <- ifelse(pe$group == 1, "Nigeria", ifelse(pe$group == 2, "Ghana", NA))
key <- pe[pe$op == "~" & pe$rhs %in% c("engagement", "HARM", "everybody"), ]
# Wald test of equality for each structural path across countries
key$wald_p_equal <- NA
for (lab in unique(sub("_NG|_GH", "", key$label[key$label != ""]))) {
  w <- lavTestWald(s_free, paste0(lab, "_NG == ", lab, "_GH"))
  key$wald_p_equal[sub("_NG|_GH", "", key$label) == lab] <- w$p.value
}
write.csv(data.frame(country = key$country, outcome = key$lhs, predictor = key$rhs, label = key$label,
                     b = round(key$est, 3), se = round(key$se, 3), p = signif(key$pvalue, 3),
                     beta = round(key$std.all, 3), ci_lo = round(key$ci.lower, 3), ci_hi = round(key$ci.upper, 3),
                     wald_p_equal = signif(key$wald_p_equal, 3)),
          file.path(tab, "T6a_sem_structural_paths.csv"), row.names = FALSE)
ctrl <- pe[pe$op == "~" & pe$rhs %in% c("age_o", "female", "city", "income_top", "student"), ]
write.csv(data.frame(country = ctrl$country, outcome = ctrl$lhs, predictor = ctrl$rhs,
                     b = round(ctrl$est, 3), se = round(ctrl$se, 3), p = signif(ctrl$pvalue, 3),
                     beta = round(ctrl$std.all, 3)), file.path(tab, "T6b_sem_control_paths.csv"), row.names = FALSE)
r2 <- lavInspect(s_free, "r2")
write.csv(data.frame(country = rep(c("Nigeria", "Ghana"), each = length(r2[[1]])),
                     outcome = c(names(r2[[1]]), names(r2[[2]])), r2 = round(c(r2[[1]], r2[[2]]), 3)),
          file.path(tab, "T6c_sem_r2.csv"), row.names = FALSE)
# omnibus test: all structural paths equal across countries
labs <- unique(sub("_NG|_GH", "", key$label[key$label != ""]))
omni <- lavTestWald(s_free, paste0(labs, "_NG == ", labs, "_GH", collapse = "\n"))
semfit <- getfit(s_free, "Multi-group SEM (partial scalar measurement)")
semfit$omnibus_wald_paths_equal <- round(omni$stat, 3); semfit$omnibus_df <- omni$df
semfit$omnibus_p <- signif(omni$p.value, 3)
# likelihood-ratio check: all structural paths constrained equal vs free (scaled difference test)
eq_syn <- gsub("c\\(([a-f][0-9])_NG,[a-f][0-9]_GH\\)\\*", "c(\\1,\\1)*", paths())
s_eq <- fit_sem(d, syn = eq_syn)
lr_eq <- lavTestLRT(s_eq, s_free, method = "satorra.bentler.2001")
semfit$lrt_chisq_diff <- round(lr_eq[2, "Chisq diff"], 3); semfit$lrt_df <- lr_eq[2, "Df diff"]
semfit$lrt_p <- signif(lr_eq[2, "Pr(>Chisq)"], 3)
semfit$delta_cfi_constrained <- round(fitMeasures(s_eq, "cfi.robust") - fitMeasures(s_free, "cfi.robust"), 4)
write.csv(semfit, file.path(tab, "T6d_sem_fit.csv"), row.names = FALSE)

# indirect effects: Monte Carlo confidence intervals (20,000 draws; Preacher & Selig, 2012)
mc <- monteCarloCI(s_free, nRep = 20000, standardized = FALSE)
mc <- data.frame(effect = rownames(mc), round(mc, 4), row.names = NULL)
dl <- pe[pe$op == ":=", c("lhs", "est", "se", "pvalue")]
mc <- merge(mc, data.frame(effect = dl$lhs, delta_se = round(dl$se, 4), delta_p = signif(dl$pvalue, 3)), by = "effect")
write.csv(mc, file.path(tab, "T6e_indirect_effects_montecarlo.csv"), row.names = FALSE)

# ------------------------------------------------------------------ 5. Robustness of the structural model
rob <- function(f, label) {
  p <- parameterEstimates(f, standardized = TRUE)
  p <- p[p$op == "~" & p$rhs %in% c("engagement", "HARM", "everybody"), ]
  data.frame(spec = label, country = ifelse(p$group == 1, "Nigeria", "Ghana"), outcome = p$lhs,
             predictor = p$rhs, beta = round(p$std.all, 3), p = signif(p$pvalue, 3))
}
R <- list(rob(s_free, "Main: MLR-FIML, partial scalar"))
# (a) exclude responses with any quality flag
R[[2]] <- rob(fit_sem(d[d$flag_any_quality == 0, ]), "Excluding quality-flagged responses")
# (b) random-intercept (acquiescence) factor: all Likert items load 1, orthogonal to substantive factors
ri_items <- c(harm, supp, "everybody", "charges", "gov_resp", "cheaper_plastic", "trash_no_harm",
              "not_recycled", "not_reused")
ri_syn <- paste(sub("SUPPORT =~", paste0("RI =~ ", paste0("1*", ri_items, collapse = " + "),
                                          "\nRI ~~ 0*HARM + 0*SUPPORT\nSUPPORT =~"), paths()), "")
ri_syn <- sub("HARM ~ ", "RI ~~ 0*engagement + 0*age_o + 0*female + 0*city + 0*income_top + 0*student\nHARM ~ ", ri_syn)
s_ri <- tryCatch(fit_sem(d, syn = ri_syn), error = function(e) NULL)
if (!is.null(s_ri)) {
  R[[3]] <- rob(s_ri, "Random-intercept response-style factor")
  sink_txt(summary(s_ri, fit.measures = TRUE, standardized = TRUE), "sem_random_intercept.txt")
  ri_var <- parameterEstimates(s_ri); ri_var <- ri_var[ri_var$lhs == "RI" & ri_var$op == "~~" & ri_var$rhs == "RI", ]
  write.csv(data.frame(country = c("Nigeria", "Ghana"), ri_variance = round(ri_var$est, 3),
                       se = round(ri_var$se, 3), p = signif(ri_var$pvalue, 3),
                       rbind(getfit(s_ri, "RI model"))[rep(1, 2), -1]),
            file.path(tab, "T7b_random_intercept_variance.csv"), row.names = FALSE)
}
# (c) ordinal WLSMV structural model: the 18 factor indicators are treated as ordered, the single-item
#     norm and stance outcomes as continuous (declaring them ordered too prevents convergence);
#     listwise deletion, as WLSMV does not use FIML
ords <- c(harm, supp)
dw <- dord
# capped at 20 minutes of elapsed time; if it does not finish, the check is reported as not run
s_w <- tryCatch({
  setTimeLimit(elapsed = 1200, transient = TRUE)
  sem(gsub("c\\([a-f][0-9]_NG,[a-f][0-9]_GH\\)\\*", "", paths()), dw, group = "country",
      ordered = intersect(ords, names(dw)), estimator = "WLSMV",
      group.equal = c("loadings", "thresholds"))
}, error = function(e) {print(e); NULL})
setTimeLimit(elapsed = Inf)
if (!is.null(s_w) && !lavInspect(s_w, "converged")) { message("Ordinal SEM did not converge; not reported"); s_w <- NULL }
if (!is.null(s_w)) R[[length(R) + 1]] <- rob(s_w, "Ordinal WLSMV (listwise)")
# (d) alternative specification: "everybody can play a role" as a fifth SUPPORT indicator (as in the
#     published single-country PCA) and no separate norm; harm and engagement predict support directly
alt_syn <- '
HARM =~ flood + erosion + vector + infiltration + buildings + toxic + microbes + nutrients +
        bags_billions + waterways_tons + aquatic + food + terrestrial + harmful_env
SUPPORT =~ everybody + polluter_pays + legal + biodeg_alt + cheap_bioplastic
bags_billions ~~ waterways_tons
aquatic ~~ terrestrial
biodeg_alt ~~ cheap_bioplastic
HARM ~ engagement + age_o + female + city + income_top + student
SUPPORT ~ HARM + engagement + age_o + female + city + income_top + student
charges ~ HARM + engagement
gov_resp ~ HARM + engagement
cheaper_plastic ~ HARM + engagement
charges ~~ gov_resp + cheaper_plastic
gov_resp ~~ cheaper_plastic
SUPPORT ~~ charges + gov_resp + cheaper_plastic
'
s_alt <- fit_sem(d, syn = alt_syn)
R[[length(R) + 1]] <- rob(s_alt, "Alternative: norm item as SUPPORT indicator")
# (e0) covariates treated as fixed (listwise on covariates)
R[[length(R) + 1]] <- rob(fit_sem(d, fx = TRUE), "Covariates fixed (fixed.x = TRUE)")
# (e) excluding the uniform-disagreement latent class identified in step 3
cls_f <- file.path(tab, "T10f_lca_assignments.csv")
if (file.exists(cls_f)) {
  cls <- read.csv(cls_f); cm <- read.csv(file.path(tab, "T10e_lca_class_means.csv"))
  uni <- cm$class[which.min(cm$SUPPORT)]
  d_ex <- d[!(d$resp_id %in% cls$resp_id[cls$class == uni]), ]
  s_ex <- fit_sem(d_ex)
  R[[length(R) + 1]] <- rob(s_ex, "Excluding uniform-disagreement class")
  # latent means and invariance without that class
  m_ex <- cfa(cfa_m, d_ex, group = "country", estimator = "MLR", missing = "fiml",
              group.equal = c("loadings", "intercepts"), group.partial = freed)
  pe_ex <- parameterEstimates(m_ex)
  lat_ex <- pe_ex[pe_ex$op == "~1" & pe_ex$lhs %in% c("HARM", "SUPPORT") & pe_ex$group == 2, ]
  write.csv(data.frame(factor = lat_ex$lhs, ghana_minus_nigeria = round(lat_ex$est, 3), se = round(lat_ex$se, 3),
                       p = signif(lat_ex$pvalue, 3), ci_lo = round(lat_ex$ci.lower, 3), ci_hi = round(lat_ex$ci.upper, 3)),
            file.path(tab, "T5d_latent_means_excl_uniform.csv"), row.names = FALSE)
  m_ex_conf <- cfa(cfa_m, d_ex, group = "country", estimator = "MLR", missing = "fiml")
  m_ex_metr <- cfa(cfa_m, d_ex, group = "country", estimator = "MLR", missing = "fiml", group.equal = "loadings")
  write.csv(rbind(getfit(m_ex_conf, "Configural (excl. uniform class)"), getfit(m_ex_metr, "Metric (excl.)"),
                  getfit(m_ex, "Scalar (excl.)")), file.path(tab, "T5e_invariance_excl_uniform.csv"), row.names = FALSE)
  # omnibus equality of structural paths without that class
  key_ex <- parameterEstimates(s_ex)
  labs_ex <- unique(sub("_NG|_GH", "", key_ex$label[key_ex$op == "~" & grepl("_NG$", key_ex$label)]))
  om_ex <- lavTestWald(s_ex, paste0(labs_ex, "_NG == ", labs_ex, "_GH", collapse = "\n"))
  write.csv(data.frame(spec = "Excluding uniform-disagreement class", wald = round(om_ex$stat, 3), df = om_ex$df,
                       p = signif(om_ex$p.value, 3)), file.path(tab, "T7e_omnibus_excl_uniform.csv"), row.names = FALSE)
}
# (f) pooled model with country dummy and country x predictor interactions is reported via Wald tests above
write.csv(do.call(rbind, R), file.path(tab, "T7a_sem_robustness.csv"), row.names = FALSE)
robfit <- rbind(getfit(s_free, "Main"), getfit(s_alt, "Alternative specification"),
                if (!is.null(s_ri)) getfit(s_ri, "Random intercept"),
                if (!is.null(s_w)) getfit(s_w, "Ordinal WLSMV"))
write.csv(robfit, file.path(tab, "T7c_sem_robustness_fit.csv"), row.names = FALSE)

# ------------------------------------------------------------------ 6. Missing-data summary
miss <- sapply(c("Nigeria", "Ghana"), function(cty) {
  dd <- d[d$country == cty, c(harm, supp, "everybody", "charges", "gov_resp", "cheaper_plastic")]
  c(any_missing_pct = mean(!complete.cases(dd)) * 100, mean_item_missing_pct = mean(is.na(dd)) * 100,
    max_item_missing_pct = max(colMeans(is.na(dd))) * 100)
})
write.csv(data.frame(metric = rownames(miss), round(miss, 2)), file.path(tab, "T7d_missingness.csv"), row.names = FALSE)
writeLines(capture.output(sessionInfo()), file.path(mod, "R_sessionInfo.txt"))
cat("SEM step complete\n")

In [ ]:
run_r("04_measurement_sem.R")
pd.read_csv(TAB / "T6a_sem_structural_paths.csv")[["country", "outcome", "predictor", "beta", "p", "wald_p_equal"]]

## Step 5. Machine learning: prediction, transfer between countries, and SHAP

**Question 1.** Can beliefs and background predict who supports each instrument? We use **nested cross-validation**: the outer folds give an honest estimate and the inner folds tune the model. We compare penalised logistic regression (the benchmark), a random forest and XGBoost.

**Question 2.** Does a model trained in one country work in the other? We train on all Nigerians and test on Ghanaians, then the reverse. The *transfer gap* is the target country's own cross-validated AUC minus the transferred AUC, with bootstrap 95% CIs. A small gap is not proof that transfer works, so we also test it against a margin of 0.05 AUC (`EQ_MARGIN`): transfer is *non-inferior* when the upper end of the 90% bootstrap interval is below 0.05, and *equivalent* when the whole 90% interval lies within ±0.05 (two one-sided tests, Lakens 2017).

**Question 3.** Does knowing the country add anything once individual beliefs are known? This is tested with a pooled model, with and without a country indicator.

**Question 4.** Which features carry the predictions? XGBoost is refitted on bootstrap samples and explained with TreeSHAP. SHAP values are summed within feature groups (harm beliefs, demographics, and so on).

Items from the responsibility block are never used as predictors, because predicting an item from its sibling items would be circular.

In [ ]:
def add_features(df):
    df = df.copy()
    df["female"] = (df.gender == "Female").astype(float).where(df.gender.notna())
    df["age_o"] = df.age.map({"18-27": 1, "28-37": 2, "38-47": 3, "48+": 4})
    df["city"] = (df.community == "City").astype(float).where(df.community.notna())
    df["hh_o"] = df.hh_size.map({"1-3": 1, "4-6": 2, "7-9": 3, "10 and above": 4})
    df["yrs_o"] = df.residence_years.map({"1-2": 1, "2-5": 2, "6-10": 3, ">10": 4})
    for grp, k in [("Student", "occ_student"), ("Academic or researcher", "occ_academic"),
                   ("Public sector", "occ_public"), ("Business or self-employed", "occ_business")]:
        df[k] = (df.occ_group == grp).astype(float).where(df.occ_group.notna())
    df["married"] = (df.marital == "Married").astype(float).where(df.marital.notna())
    df["stem"] = df.discipline.isin(["Science", "Engineering and Technology", "Agriculture",
                                     "Medical and Health Sciences"]).astype(float).where(df.discipline.notna())
    return df

GROUPS = {"Demographics": ["female", "age_o", "married", "city", "hh_o", "yrs_o", "occ_student", "occ_academic",
                           "occ_public", "occ_business", "stem", "income_top"],
          "Engagement and awareness": ["env_gathering", "heard_ce", "heard_bioplastic"],
          "Economic beliefs": ["waste_wealth", "bioplastic_benefit"],
          "Plastic use": ["use_bag", "use_bottle", "use_pvc", "use_styrofoam"],
          "Harm beliefs": HARM,
          "Recycling beliefs": ["trash_no_harm", "not_recycled", "not_reused"]}
FEATS = [f for fs in GROUPS.values() for f in fs]
FEAT_GROUP = {f: grp for grp, fs in GROUPS.items() for f in fs}
LAB = dict(LABEL, female="Female", age_o="Age group", married="Married", city="Lives in city", hh_o="Household size",
           yrs_o="Years in state/region", occ_student="Student", occ_academic="Academic/researcher",
           occ_public="Public sector", occ_business="Business/self-employed", stem="STEM discipline",
           income_top="Highest income band", env_gathering="Attended environmental event",
           heard_ce="Heard of circular economy", heard_bioplastic="Heard of bioplastic",
           waste_wealth="Waste can generate wealth", bioplastic_benefit="Bioplastics beneficial",
           use_bag="Uses plastic bags", use_bottle="Uses plastic bottles", use_pvc="Uses PVC", use_styrofoam="Uses Styrofoam")
OUTCOMES = {"polluter_pays": "Polluter pays", "legal": "Legal action", "charges": "Extra charges",
            "biodeg_alt": "Biodegradable alternatives", "gov_resp": "Government responsibility",
            "cheaper_plastic": "Make plastic cheaper", "pref_burning": "Prefers burning"}
print(len(FEATS), "features in", len(GROUPS), "groups")

def target(df, o):
    # 1 = agrees (4 or 5 on the scale); for burning, 1 = ticked burning as a preferred method
    y = (df[o] == 1) if o == "pref_burning" else (df[o] >= 4)
    return y.astype(int).where(df[o].notna())

def models(with_rf=True):
    # Small tuning grids keep the run time reasonable; the random forest uses fixed settings
    m = {"Logistic (L2)": (make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=5000)),
                           {"logisticregression__C": [0.05, 0.5]}),
         "Random forest": (make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(
                               n_estimators=300, min_samples_leaf=10, max_features="sqrt", random_state=SEED, n_jobs=1)), None),
         "XGBoost": (xgb.XGBClassifier(n_estimators=200, learning_rate=0.05, subsample=0.8, colsample_bytree=0.7,
                                       reg_lambda=5, eval_metric="logloss", random_state=SEED, n_jobs=1, verbosity=0),
                     {"max_depth": [2, 3], "min_child_weight": [1, 5]})}
    if not with_rf:
        m.pop("Random forest")
    return m

def tuned(model, grid, X, y, seed):
    if grid is None:
        return clone(model).fit(X, y), {}
    gs = GridSearchCV(model, grid, scoring="roc_auc", cv=StratifiedKFold(3, shuffle=True, random_state=seed), n_jobs=1)
    gs.fit(X, y)
    return gs.best_estimator_, gs.best_params_

BOOT_DIFF = []   # keeps the last bootstrap distribution of AUC differences (used for the equivalence test)

def boot_auc(y, p, B, rng, p2=None):
    # Bootstrap 95% CI for AUC; with p2, also for the AUC difference (p2 minus p) on the same people
    y, p = np.asarray(y), np.asarray(p); out, diff = [], []
    for _ in range(B):
        i = rng.integers(0, len(y), len(y))
        if y[i].min() == y[i].max():
            continue
        a = roc_auc_score(y[i], p[i]); out.append(a)
        if p2 is not None:
            diff.append(roc_auc_score(y[i], np.asarray(p2)[i]) - a)
    ci = (np.percentile(out, 2.5), np.percentile(out, 97.5))
    BOOT_DIFF[:] = diff
    return ci if p2 is None else (ci, (np.percentile(diff, 2.5), np.percentile(diff, 97.5)))

def calib(y, p):
    # Calibration-in-the-large (observed minus predicted rate) and calibration slope
    lp = np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
    m = LogisticRegression(C=1e6, max_iter=1000).fit(lp.reshape(-1, 1), y)
    return float(np.mean(y) - np.mean(p)), float(m.coef_[0, 0])

In [ ]:
def within_and_transfer(df, rng, with_rf=True):
    # Nested CV within each country, then transfer to the other country, for every outcome and model
    perf_rows, trans_rows, oof = [], [], {}
    for o, olab in OUTCOMES.items():
        for cty in COUNTRIES:
            sub = df[df.country == cty].copy(); sub["y"] = target(sub, o); sub = sub[sub.y.notna()]
            X, y = sub[FEATS].values, sub.y.astype(int).values
            outer = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=SEED)
            for mname, (m, grid) in models(with_rf).items():
                pred, fold_auc = np.zeros((2, len(y))), []
                for k, (tr, te) in enumerate(outer.split(X, y)):
                    est, _ = tuned(clone(m), grid, X[tr], y[tr], SEED + k)
                    pred[k // 5, te] = est.predict_proba(X[te])[:, 1]
                    fold_auc.append(roc_auc_score(y[te], pred[k // 5, te]))
                p = pred.mean(0); oof[(o, cty, mname)] = (y, p)
                lo, hi = boot_auc(y, p, N_BOOT_AUC, rng)
                perf_rows.append({"outcome": olab, "country": cty, "model": mname, "n": len(y), "prevalence": y.mean(),
                                  "auc_oof": roc_auc_score(y, p), "auc_lo": lo, "auc_hi": hi,
                                  "auc_fold_mean": np.mean(fold_auc), "auc_fold_sd": np.std(fold_auc),
                                  "pr_auc": average_precision_score(y, p), "brier": brier_score_loss(y, p),
                                  "brier_null": brier_score_loss(y, np.full(len(y), y.mean()))})
        for src, tgt in [("Nigeria", "Ghana"), ("Ghana", "Nigeria")]:
            S, T = df[df.country == src].copy(), df[df.country == tgt].copy()
            S["y"], T["y"] = target(S, o), target(T, o); S, T = S[S.y.notna()], T[T.y.notna()]
            for mname, (m, grid) in models(with_rf).items():
                est, _ = tuned(clone(m), grid, S[FEATS].values, S.y.astype(int).values, SEED)
                pt, yt = est.predict_proba(T[FEATS].values)[:, 1], T.y.astype(int).values
                y_own, p_own = oof[(o, tgt, mname)]
                (lo, hi), (dlo, dhi) = boot_auc(yt, pt, N_BOOT_AUC, rng, p2=p_own)
                g90 = np.percentile(BOOT_DIFF, [5, 95])      # 90% interval = two one-sided tests at 5%
                citl, slope = calib(yt, pt)
                trans_rows.append({"outcome": olab, "source": src, "target": tgt, "model": mname,
                                   "auc_transfer": roc_auc_score(yt, pt), "auc_lo": lo, "auc_hi": hi,
                                   "auc_target_own_cv": roc_auc_score(y_own, p_own),
                                   "transfer_gap": roc_auc_score(y_own, p_own) - roc_auc_score(yt, pt),
                                   "gap_lo": dlo, "gap_hi": dhi, "gap_lo90": g90[0], "gap_hi90": g90[1],
                                   "equivalent_within_0.05": bool(g90[0] > -EQ_MARGIN and g90[1] < EQ_MARGIN),
                                   "noninferior_0.05": bool(g90[1] < EQ_MARGIN),    # transfer loses less than the margin
                                   "calib_in_large": citl, "calib_slope": slope,
                                   "prev_source": S.y.mean(), "prev_target": yt.mean()})
        print(olab, "done", flush=True)
    return pd.DataFrame(perf_rows), pd.DataFrame(trans_rows)

dm = add_features(d)
rng_ml = np.random.default_rng(SEED)
T8a, T8b = within_and_transfer(dm, rng_ml)
T8a.round(4).to_csv(TAB / "T8a_ml_within_country_nestedcv.csv", index=False)
T8b.round(4).to_csv(TAB / "T8b_ml_cross_country_transfer.csv", index=False)
T8a.pivot_table(index="outcome", columns=["country", "model"], values="auc_oof").round(3)

In [ ]:
# Pooled data: does adding a country indicator improve prediction? (penalised logistic regression)
rows = []
for o, olab in OUTCOMES.items():
    sub = dm.copy(); sub["y"] = target(sub, o); sub = sub[sub.y.notna()]
    sub["country_ghana"] = (sub.country == "Ghana").astype(int); y = sub.y.astype(int).values
    res = {}
    for spec, cols in [("features", FEATS), ("features + country", FEATS + ["country_ghana"]), ("country only", ["country_ghana"])]:
        m, grid = models()["Logistic (L2)"]; pred = np.zeros(len(y))
        for k, (tr, te) in enumerate(StratifiedKFold(5, shuffle=True, random_state=SEED).split(sub, y)):
            est, _ = tuned(clone(m), grid, sub[cols].values[tr], y[tr], SEED + k)
            pred[te] = est.predict_proba(sub[cols].values[te])[:, 1]
        res[spec] = pred
    _, (dlo, dhi) = boot_auc(y, res["features"], N_BOOT_AUC, rng_ml, p2=res["features + country"])
    rows.append({"outcome": olab, "auc_features": roc_auc_score(y, res["features"]),
                 "auc_features_country": roc_auc_score(y, res["features + country"]),
                 "auc_country_only": roc_auc_score(y, res["country only"]),
                 "delta_auc_country": roc_auc_score(y, res["features + country"]) - roc_auc_score(y, res["features"]),
                 "delta_lo": dlo, "delta_hi": dhi})
T8c = pd.DataFrame(rows); T8c.round(4).to_csv(TAB / "T8c_ml_pooled_country_increment.csv", index=False)
T8c.round(3)

In [ ]:
# Bootstrap SHAP for the four main instruments (XGBoost)
SHAP_OUT = ["polluter_pays", "legal", "charges", "biodeg_alt"]
shap_rows, sim_rows, group_rows, dep_rows = [], [], [], []
for o in SHAP_OUT:
    imp = {}
    for cty in COUNTRIES:
        sub = dm[dm.country == cty].copy(); sub["y"] = target(sub, o); sub = sub[sub.y.notna()]
        X, y = sub[FEATS], sub.y.astype(int).values
        m, grid = models()["XGBoost"]
        _, bp = tuned(clone(m), grid, X.values, y, SEED)
        base = clone(m).set_params(**bp)
        B, G = np.zeros((N_BOOT_SHAP, len(FEATS))), np.zeros((N_BOOT_SHAP, len(GROUPS)))
        harm_shap = np.zeros(len(y))
        rng = np.random.default_rng(SEED + 100 * SHAP_OUT.index(o) + COUNTRIES.index(cty))
        for b in range(N_BOOT_SHAP):
            i = rng.integers(0, len(y), len(y))
            mb = clone(base).set_params(random_state=SEED + b).fit(X.values[i], y[i])
            sv = shap.TreeExplainer(mb).shap_values(X.values)          # explain the original sample
            B[b] = np.abs(sv).mean(0)
            for gi, fs in enumerate(GROUPS.values()):
                G[b, gi] = np.abs(sv[:, [FEATS.index(f) for f in fs]].sum(1)).mean()   # grouped SHAP
            harm_shap += sv[:, [FEATS.index(f) for f in HARM]].sum(1) / N_BOOT_SHAP
        dep_rows.append(pd.DataFrame({"outcome": OUTCOMES[o], "country": cty, "harm_mean": sub[HARM].mean(1).values,
                                      "harm_shap_sum": harm_shap}))
        imp[cty] = B
        ranks = (-B).argsort(1).argsort(1) + 1
        for j, f in enumerate(FEATS):
            shap_rows.append({"outcome": OUTCOMES[o], "country": cty, "feature": f, "label": LAB[f], "group": FEAT_GROUP[f],
                              "mean_abs_shap": B[:, j].mean(), "lo": np.percentile(B[:, j], 2.5), "hi": np.percentile(B[:, j], 97.5),
                              "median_rank": np.median(ranks[:, j]), "rank_lo": np.percentile(ranks[:, j], 2.5),
                              "rank_hi": np.percentile(ranks[:, j], 97.5), "p_top10": np.mean(ranks[:, j] <= 10)})
        for gi, grp in enumerate(GROUPS):
            group_rows.append({"outcome": OUTCOMES[o], "country": cty, "group": grp, "mean_abs_group_shap": G[:, gi].mean(),
                               "lo": np.percentile(G[:, gi], 2.5), "hi": np.percentile(G[:, gi], 97.5)})
    # are the importance profiles more alike within a country (bootstrap replicates) than across countries?
    rng = np.random.default_rng(SEED)
    cross = [stats.spearmanr(imp["Nigeria"][rng.integers(N_BOOT_SHAP)], imp["Ghana"][rng.integers(N_BOOT_SHAP)])[0] for _ in range(2000)]
    within = {c: [stats.spearmanr(*imp[c][rng.choice(N_BOOT_SHAP, 2, replace=False)])[0] for _ in range(2000)] for c in COUNTRIES}
    top = {c: set(np.array(FEATS)[imp[c].mean(0).argsort()[::-1][:10]]) for c in COUNTRIES}
    sim_rows.append({"outcome": OUTCOMES[o], "rho_mean_importance": stats.spearmanr(imp["Nigeria"].mean(0), imp["Ghana"].mean(0))[0],
                     "rho_cross_median": np.median(cross), "rho_cross_lo": np.percentile(cross, 2.5), "rho_cross_hi": np.percentile(cross, 97.5),
                     "rho_within_NG_median": np.median(within["Nigeria"]), "rho_within_GH_median": np.median(within["Ghana"]),
                     "top10_shared_n": len(top["Nigeria"] & top["Ghana"]),
                     "top10_shared": "; ".join(sorted(LAB[f] for f in top["Nigeria"] & top["Ghana"]))})
    print("SHAP", OUTCOMES[o], "done", flush=True)
T9a, T9b, T9c = pd.DataFrame(shap_rows), pd.DataFrame(sim_rows), pd.DataFrame(group_rows)
T9a.round(5).to_csv(TAB / "T9a_shap_bootstrap_importance.csv", index=False)
T9b.round(4).to_csv(TAB / "T9b_shap_cross_country_similarity.csv", index=False)
T9c.round(5).to_csv(TAB / "T9c_shap_group_importance.csv", index=False)
pd.concat(dep_rows).round(5).to_csv(TAB / "T9d_shap_harm_dependence.csv", index=False)
T9b.round(3)

In [ ]:
# Sensitivity: repeat prediction and transfer without the uniform-disagreement class (logistic and XGBoost)
dm_ex = dm[~dm.resp_id.isin(cls.loc[cls["class"] == UNIFORM, "resp_id"])]
T8a_ex, T8b_ex = within_and_transfer(dm_ex, np.random.default_rng(SEED), with_rf=False)
T8a_ex.round(4).to_csv(TAB / "T8a_ml_within_country_nestedcv_excl_uniform.csv", index=False)
T8b_ex.round(4).to_csv(TAB / "T8b_ml_cross_country_transfer_excl_uniform.csv", index=False)
T8a_ex.pivot_table(index="outcome", columns=["country", "model"], values="auc_oof").round(3)

## Step 6. Open-ended answers: how can plastic waste create wealth?

Answers are coded with a transparent keyword codebook (12 themes, `docs/text_codebook.md`), so every code can be traced to a pattern. Eight themes name a **concrete pathway** (products, energy, construction, collection and sale, jobs, enterprise, crafts, agriculture). We count the concrete pathways in each answer. A random sample of 120 answers is exported for two human coders, to check the codebook before publication.

In [ ]:
t = d[d.wealth_text.notna()].copy()
t["text"] = t.wealth_text.str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
nonsub = r"^(yes|yes\.|no|nil|none|nothing|ok|okay|n/?a|no idea|i don'?t know|not sure|\W*)$"
t["substantive"] = ~t.text.str.match(nonsub) & (t.text.str.len() > 3)
CODEBOOK = {
    "Recycling (generic)": r"recycl|recyc|recyl|re-cycl|reprocess|re-process|regenerat|renew",
    "Reuse and upcycling": r"\breus|re-us|repurpos|refurbish|upcycl|up-cycl|up cycl|repair",
    "New consumer products": r"product|chair|bucket|bowl|table|furniture|sandal|cloth|textile|fabric|toy|container|\bcups?\b|\bmats?\b|plate|\bitems?\b|basket|bottle",
    "Energy and fuel": r"energy|fuel|\boil\b|\bgas\b|biogas|syngas|pyroly|gasifi|electric|\bpower\b|diesel|petrol|combust|incinerat",
    "Construction materials": r"brick|block|\broads?\b|pave|tile|building|construct|roof|cement|concrete|asphalt",
    "Collection and sale": r"\bsell|\bsale|\bsold|\bbuy|collect|gather|scaveng|\btrade|export|buy.?back|exchange|pick",
    "Jobs and livelihoods": r"\bjobs?\b|employ|livelihood|work for|unemploy|youth",
    "Enterprise and industry": r"business|entrepreneur|company|companies|industr|factor(y|ies)|venture|profit|revenue|invest|start.?up|sme|manufactur",
    "Crafts and art": r"\bart\b|artistic|craft|decorat|fashion|design|bead|handwork|ornament",
    "Agriculture": r"farm|agric|nursery|plant pot|compost|seedling",
    "Environmental and health co-benefits": r"environment|clean|pollution|flood|hygiene|health|drain|sanitat",
    "Governance and education": r"government|policy|policies|\blaw|incentive|subsid|deposit|enforce|regulat|awareness|educat|sensiti[sz]|campaign|train"}
CONCRETE = ["New consumer products", "Energy and fuel", "Construction materials", "Collection and sale",
            "Jobs and livelihoods", "Enterprise and industry", "Crafts and art", "Agriculture"]
for code_, pat in CODEBOOK.items():
    t[code_] = t.text.str.contains(pat, regex=True).astype(int) * t.substantive.astype(int)
t = t.copy()
t["n_specific_pathways"] = t[CONCRETE].sum(axis=1)
t["generic_only"] = ((t.n_specific_pathways == 0) & t.substantive).astype(int)
t["word_count"] = t.text.str.split().str.len()
s = t[t.substantive]

rows = []
for code_ in list(CODEBOOK) + ["generic_only"]:
    a, b = s.loc[s.country == "Nigeria", code_], s.loc[s.country == "Ghana", code_]
    ct = np.array([[a.sum(), len(a) - a.sum()], [b.sum(), len(b) - b.sum()]])
    p = stats.fisher_exact(ct)[1] if (stats.contingency.expected_freq(ct) < 5).any() else stats.chi2_contingency(ct, correction=False)[1]
    ca, cb = proportion_confint(a.sum(), len(a), method="wilson"), proportion_confint(b.sum(), len(b), method="wilson")
    rows.append({"code": code_, "Nigeria_n": int(a.sum()), "Nigeria_pct": 100 * a.mean(), "NG_lo": 100 * ca[0], "NG_hi": 100 * ca[1],
                 "Ghana_n": int(b.sum()), "Ghana_pct": 100 * b.mean(), "GH_lo": 100 * cb[0], "GH_hi": 100 * cb[1], "p": p})
T11a = pd.DataFrame(rows); T11a["q_bh"] = multipletests(T11a.p, method="fdr_bh")[1]
T11a.round(4).to_csv(TAB / "T11a_text_code_prevalence.csv", index=False)
u = stats.mannwhitneyu(s.loc[s.country == "Ghana", "n_specific_pathways"], s.loc[s.country == "Nigeria", "n_specific_pathways"])
uw = stats.mannwhitneyu(s.loc[s.country == "Ghana", "word_count"], s.loc[s.country == "Nigeria", "word_count"])
T11b = pd.DataFrame([{"country": c, "respondents": int((d.country == c).sum()), "answered": int((t.country == c).sum()),
                      "substantive": int((s.country == c).sum()), "median_words": s.loc[s.country == c, "word_count"].median(),
                      "mean_specific_pathways": s.loc[s.country == c, "n_specific_pathways"].mean(),
                      "pct_two_plus_pathways": 100 * (s.loc[s.country == c, "n_specific_pathways"] >= 2).mean(),
                      "mw_p_specificity": u.pvalue, "mw_p_wordcount": uw.pvalue} for c in COUNTRIES])
T11b.round(4).to_csv(TAB / "T11b_text_summary.csv", index=False)

# Keyness: words that best separate the two countries (weighted log-odds, Monroe et al. 2008)
STOP = set('''a an the and or of to in into on for by with from as at is are be been being it its this that these those can could will
would may might should must do does did done have has had having i we you they he she them our their your my me us so if than then there
here which who whom what when where how why not no yes also very more most such other others some any all each every just only own same
both through via about over under up down out again further once much many well use used using make made making way ways able get getting
gets one two waste wastes plastic plastics generate generating wealth'''.split())
tok = lambda x: [w for w in re.findall(r"[a-z]+", x) if w not in STOP and len(w) > 2]
cnt = {c: Counter(w for x in s.loc[s.country == c, "text"] for w in set(tok(x))) for c in COUNTRIES}
vocab = [w for w in set(cnt["Nigeria"]) | set(cnt["Ghana"]) if cnt["Nigeria"][w] + cnt["Ghana"][w] >= 5]
prior = cnt["Nigeria"] + cnt["Ghana"]; a0 = sum(prior[w] for w in vocab)
n1, n2 = sum(cnt["Ghana"][w] for w in vocab), sum(cnt["Nigeria"][w] for w in vocab); sc = 0.5 * (n1 + n2) / a0
kw = []
for w in vocab:
    aw, y1, y2 = prior[w] * sc, cnt["Ghana"][w], cnt["Nigeria"][w]
    l1 = np.log((y1 + aw) / (n1 + a0 * sc - y1 - aw)); l2 = np.log((y2 + aw) / (n2 + a0 * sc - y2 - aw))
    kw.append({"word": w, "Ghana_docs": y1, "Nigeria_docs": y2, "z": (l1 - l2) / np.sqrt(1 / (y1 + aw) + 1 / (y2 + aw))})
pd.DataFrame(kw).sort_values("z").round(4).to_csv(TAB / "T11c_text_keyness.csv", index=False)

# Validation sample for two human coders (fill coder1:: and coder2:: columns with 0/1)
val = pd.concat([t[t.substantive & (t.country == c)].sample(60, random_state=SEED) for c in COUNTRIES])
val = val[["resp_id", "country", "wealth_text"] + list(CODEBOOK)].rename(columns={c: f"auto::{c}" for c in CODEBOOK})
for c in CODEBOOK:
    val[f"coder1::{c}"] = ""; val[f"coder2::{c}"] = ""
val.sample(frac=1, random_state=SEED).to_csv(ROOT / "docs" / "text_validation_sample_for_coders.csv", index=False)
t[["resp_id", "country", "substantive", "n_specific_pathways", "generic_only"] + list(CODEBOOK)].to_csv(TAB / "T11d_text_coded_responses.csv", index=False)
T11b.round(3)

In [ ]:
# Figure 7: wealth pathways (a) and number of concrete pathways per answer (b)
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.9), layout="constrained", gridspec_kw={"width_ratios": [1.55, 1]})
ax = axes[0]
c2 = T11a[T11a.code != "generic_only"].sort_values("Nigeria_pct").reset_index(drop=True)
for cty, off, pre in [("Nigeria", 0.16, "NG"), ("Ghana", -0.16, "GH")]:
    m = c2[f"{cty}_pct"]
    ax.errorbar(m, c2.index + off, xerr=[m - c2[f"{pre}_lo"], c2[f"{pre}_hi"] - m], fmt="o", ms=3, color=COL[cty],
                elinewidth=0.7, capsize=1.2, label=cty)
ax.set_yticks(c2.index); ax.set_yticklabels([f"{c} {star(q)}" for c, q in zip(c2.code, c2.q_bh)], fontsize=6.4)
ax.set_xlabel("Answers mentioning the theme (%), Wilson 95% CI"); ax.grid(axis="x", lw=0.3, alpha=0.5)
ax.set_title("a) Wealth pathways named", loc="left")
ax = axes[1]
cats = ["0 (generic only)", "1", "2", "3 or more"]
for i, cty in enumerate(COUNTRIES):
    v = [100 * (s.loc[s.country == cty, "n_specific_pathways"].clip(upper=3) == k).mean() for k in range(4)]
    left = 0
    for k in range(4):
        ax.barh(i, v[k], left=left, color=plt.cm.Blues(0.25 + 0.22 * k), edgecolor="white", lw=0.5,
                label=cats[k] if i == 0 else None)
        if v[k] >= 8:
            ax.text(left + v[k] / 2, i, f"{v[k]:.0f}", ha="center", va="center", fontsize=6.5, color="black" if k < 2 else "white")
        left += v[k]
ax.set_yticks([0, 1]); ax.set_yticklabels([f"{c}\n(n = {(s.country == c).sum()})" for c in COUNTRIES])
ax.invert_yaxis(); ax.set_xlim(0, 100); ax.set_xlabel("Answers (%)")
ax.set_title("b) Concrete pathways per answer", loc="left")
h1, l1 = axes[0].get_legend_handles_labels(); h2, l2 = axes[1].get_legend_handles_labels()
fig.legend(h1 + h2, l1 + [f"Pathways: {x}" for x in l2], loc="outside lower center", ncol=3)
savefig(fig, "Fig7_text_wealth_pathways")

## Step 7. Figures for the structural model, latent classes and machine learning

In [ ]:
# Figure 3: the value-belief-norm chain (standardised paths, Nigeria | Ghana)
P = pd.read_csv(TAB / "T6a_sem_structural_paths.csv"); R2 = pd.read_csv(TAB / "T6c_sem_r2.csv")
def path_lab(outcome, pred):
    s_ = P[(P.outcome == outcome) & (P.predictor == pred)].set_index("country")
    sig = lambda p: "***" if p < .001 else "**" if p < .01 else "*" if p < .05 else ""
    return f"{s_.loc['Nigeria','beta']:.2f}{sig(s_.loc['Nigeria','p'])} | {s_.loc['Ghana','beta']:.2f}{sig(s_.loc['Ghana','p'])}"
def r2_lab(v):
    s_ = R2[R2.outcome == v].set_index("country").r2
    return f"R² = {s_['Nigeria']:.2f} | {s_['Ghana']:.2f}"

fig, ax = plt.subplots(figsize=(7.2, 3.0), layout="constrained")
ax.set_xlim(0, 124); ax.set_ylim(4, 56); ax.axis("off")
N = {"engagement": (9, 18, "Engagement\n(0-3 index)", "box", 16, 12),
     "HARM": (50, 18, "Harm perception\n(14 items)", "oval", 26, 14),
     "everybody": (78, 45, "Shared-\nresponsibility\nnorm", "box", 20, 14),
     "SUPPORT": (108, 18, "Support for\nregulation and\nsubstitutes (4 items)", "oval", 28, 16)}
for k, (x0, y0, txt, kind, w, h) in N.items():
    patch = (FancyBboxPatch((x0 - w / 2, y0 - h / 2), w, h, boxstyle="round,pad=0.4", fc="#f4f4f4", ec="#333", lw=0.7)
             if kind == "box" else Ellipse((x0, y0), w, h, fc="#e8f0f8", ec="#333", lw=0.7))
    ax.add_patch(patch); ax.text(x0, y0, txt, ha="center", va="center", fontsize=6.6)
    if k != "engagement":
        ax.text(x0, y0 - h / 2 - 3, r2_lab(k), ha="center", va="top", fontsize=6, color="#555")
def arrow(a, b, txt, dx=0, dy=0):
    (xa, ya, *_ , wa, ha), (xb, yb, *_, wb, hb) = N[a], N[b]
    v = np.array([xb - xa, yb - ya], float); u = v / np.linalg.norm(v)
    rad = lambda w, h: 1 / np.sqrt((u[0] / (w / 2 + 1.2)) ** 2 + (u[1] / (h / 2 + 1.2)) ** 2)
    p0, p1 = np.array([xa, ya]) + u * rad(wa, ha), np.array([xb, yb]) - u * rad(wb, hb)
    ax.add_patch(FancyArrowPatch(p0, p1, arrowstyle="-|>", mutation_scale=8, lw=1.0, color="#222"))
    m = (p0 + p1) / 2
    ax.text(m[0] + dx, m[1] + dy, txt, ha="center", va="center", fontsize=6.6, bbox=dict(fc="white", ec="none", pad=0.6))
arrow("engagement", "HARM", path_lab("HARM", "engagement"), dy=4.0)
arrow("HARM", "everybody", path_lab("everybody", "HARM"), dx=-9, dy=2)
arrow("everybody", "SUPPORT", path_lab("SUPPORT", "everybody"), dx=9, dy=2)
arrow("HARM", "SUPPORT", path_lab("SUPPORT", "HARM"), dy=-4.5)
ax.set_title("Standardised paths, Nigeria | Ghana", loc="left")
savefig(fig, "Fig3_sem_paths")

In [ ]:
# Figure 4: latent classes (a) and their share in each country (b)
ip = pd.read_csv(TAB / "T10b_lca_item_probabilities.csv"); comp = pd.read_csv(TAB / "T10c_lca_country_composition.csv")
IND = {"HARM": "Harm\nperception", "everybody": "Shared\nresponsibility", "SUPPORT": "Regulation and\nsubstitutes",
       "charges": "Extra\ncharges", "gov_resp": "Government\nresponsibility", "cheaper_plastic": "Make plastic\ncheaper",
       "engagement": "Engagement"}
CPAL = {TOP: "#1b9e77", MID: "#7570b3", UNIFORM: "#d95f02"}
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.4), layout="constrained", gridspec_kw={"width_ratios": [2.2, 1]})
ax = axes[0]
for c in [TOP, MID, UNIFORM]:
    s_ = ip[ip["class"] == c].set_index("indicator").loc[list(IND)]
    size = comp.set_index("class").loc[c, "class_size_pct"]
    ax.plot(range(len(IND)), s_.p_high, "-o", ms=3.2, lw=1.2, color=CPAL[c], label=f"{CLASS_NAME[c]} ({size:.0f}%)")
    ax.plot(range(len(IND)), s_.p_low, ":", lw=0.9, color=CPAL[c])
ax.set_xticks(range(len(IND))); ax.set_xticklabels(list(IND.values()), fontsize=6.3)
ax.set_ylim(-0.02, 1.02); ax.set_ylabel("Probability")
ax.set_title("a) Probability of high (solid) and low (dotted) category", loc="left")
ax = axes[1]; left = np.zeros(2)
for c in [TOP, MID, UNIFORM]:
    r = comp.set_index("class").loc[c]; v = np.array([r.Nigeria_pct, r.Ghana_pct])
    ax.barh([0, 1], v, left=left, color=CPAL[c], edgecolor="white", lw=0.5)
    for i in range(2):
        ax.text(left[i] + v[i] / 2, i, f"{v[i]:.0f}", ha="center", va="center", fontsize=6.5, color="white")
    left += v
ax.set_yticks([0, 1]); ax.set_yticklabels([f"Nigeria\n(n = {comp.Nigeria_n.sum()})", f"Ghana\n(n = {comp.Ghana_n.sum()})"])
ax.invert_yaxis(); ax.set_xlim(0, 100); ax.set_xlabel("Respondents (%)")
ax.set_title("b) Class shares by country", loc="left")
fig.legend(*axes[0].get_legend_handles_labels(), loc="outside lower center", ncol=3)
savefig(fig, "Fig4_latent_classes")

In [ ]:
# Figure 5: AUC within country (circles) and after transfer from the other country (squares)
OUTS = list(OUTCOMES.values()); MCOL = {"Logistic (L2)": "#4d4d4d", "XGBoost": "#6a3d9a"}
fig, axes = plt.subplots(1, 2, figsize=(7.2, 4.0), sharey=True, layout="constrained")
for ax, tgt in zip(axes, COUNTRIES):
    ax.axhspan(3.5, 6.6, color="#f2f2f2", zorder=0)
    for k, mname in enumerate(MCOL):
        w = T8a[(T8a.country == tgt) & (T8a.model == mname)].set_index("outcome").loc[OUTS]
        tr = T8b[(T8b.target == tgt) & (T8b.model == mname)].set_index("outcome").loc[OUTS]
        y0 = np.arange(len(OUTS)) + (k - 0.5) * 0.38
        ax.errorbar(w.auc_oof, y0 - 0.08, xerr=[w.auc_oof - w.auc_lo, w.auc_hi - w.auc_oof], fmt="o", ms=3.6,
                    color=MCOL[mname], elinewidth=0.8, capsize=1.5)
        ax.errorbar(tr.auc_transfer, y0 + 0.08, xerr=[tr.auc_transfer - tr.auc_lo, tr.auc_hi - tr.auc_transfer], fmt="s",
                    ms=3.6, mfc="white", color=MCOL[mname], elinewidth=0.8, capsize=1.5)
    ax.axvline(0.5, color="black", lw=0.5, ls=":"); ax.set_xlim(0.38, 0.92)
    ax.set_yticks(range(len(OUTS))); ax.set_yticklabels(OUTS); ax.set_xlabel("AUC (95% bootstrap CI)")
    ax.set_title(f"{'ab'[COUNTRIES.index(tgt)]}) Predicting respondents in {tgt}", loc="left"); ax.grid(axis="x", lw=0.3, alpha=0.5)
axes[0].set_ylim(6.6, -0.6)
handles = [Line2D([], [], color=MCOL[m], marker=mk, ms=4, mfc=f, ls="", label=f"{m}, {lab}")
           for m in MCOL for mk, f, lab in [("o", MCOL[m], "trained in that country"), ("s", "white", "transferred from the other")]]
fig.legend(handles=handles, loc="outside lower center", ncol=2)
savefig(fig, "Fig5_ml_within_vs_transfer")

In [ ]:
# Figure 6: grouped SHAP (a) and similarity of single-feature rankings (b)
GORDER = ["Harm beliefs", "Demographics", "Recycling beliefs", "Engagement and awareness", "Economic beliefs", "Plastic use"]
fig = plt.figure(figsize=(7.2, 5.0), layout="constrained")
top_sf, bot_sf = fig.subfigures(2, 1, height_ratios=[1.35, 1], hspace=0.04)
axs = top_sf.subplots(1, 4, sharey=True)
for j, o in enumerate([OUTCOMES[k] for k in SHAP_OUT]):
    ax, gg = axs[j], T9c[T9c.outcome == o]
    for k, cty in enumerate(COUNTRIES):
        s_ = gg[gg.country == cty].set_index("group").loc[GORDER]
        yy = np.arange(len(GORDER)) + (-0.19 if k == 0 else 0.19)
        ax.barh(yy, s_.mean_abs_group_shap, height=0.36, color=COL[cty], label=cty)
        ax.errorbar(s_.mean_abs_group_shap, yy, xerr=[s_.mean_abs_group_shap - s_.lo, s_.hi - s_.mean_abs_group_shap],
                    fmt="none", ecolor="black", elinewidth=0.5, capsize=1)
    ax.set_yticks(range(len(GORDER))); ax.set_yticklabels(GORDER, fontsize=6.4); ax.set_ylim(len(GORDER) - 0.4, -0.6)
    ax.set_xlim(0, 1.95); ax.set_title(o, fontsize=7.2, loc="left"); ax.set_xlabel("Mean |SHAP|", fontsize=6.5)
top_sf.suptitle("a) Importance of feature groups", x=0.01, ha="left", fontsize=8.5)
top_sf.legend(*axs[0].get_legend_handles_labels(), loc="outside lower center", ncol=2)
ax = bot_sf.subplots()
so = [OUTCOMES[k] for k in SHAP_OUT]; s_ = T9b.set_index("outcome").loc[so]; y = np.arange(len(so))
ax.errorbar(s_.rho_cross_median, y, xerr=[s_.rho_cross_median - s_.rho_cross_lo, s_.rho_cross_hi - s_.rho_cross_median],
            fmt="D", ms=4, color="black", elinewidth=0.8, capsize=1.5, label="Nigeria vs Ghana")
ax.plot(s_.rho_within_NG_median, y - 0.2, "o", ms=4, color=COL["Nigeria"], label="Within Nigeria")
ax.plot(s_.rho_within_GH_median, y + 0.2, "o", ms=4, color=COL["Ghana"], label="Within Ghana")
ax.set_yticks(y); ax.set_yticklabels(so, fontsize=6.6); ax.invert_yaxis(); ax.set_xlim(0, 1)
ax.set_xlabel(f"Spearman correlation between {len(FEATS)}-feature importance profiles"); ax.grid(axis="x", lw=0.3, alpha=0.5)
bot_sf.suptitle("b) How similar are the single-feature rankings?", x=0.01, ha="left", fontsize=8.5)
bot_sf.legend(*ax.get_legend_handles_labels(), loc="outside lower center", ncol=3)
savefig(fig, "Fig6_shap_importance")

In [ ]:
# Figure S2: key structural paths across the sensitivity checks
Rb = pd.read_csv(TAB / "T7a_sem_robustness.csv")
keys = [("everybody", "HARM", "Harm →\nnorm"), ("SUPPORT", "everybody", "Norm →\nsupport"), ("SUPPORT", "HARM", "Harm →\nsupport"),
        ("HARM", "engagement", "Engagement →\nharm"), ("charges", "HARM", "Harm →\ncharges"), ("cheaper_plastic", "HARM", "Harm →\nmake cheaper")]
specs = Rb.spec.unique().tolist()
fig, axes = plt.subplots(1, len(keys), figsize=(7.2, 3.0), sharey=True, layout="constrained")
for ax, (o, p, ttl) in zip(axes, keys):
    for i, sp in enumerate(specs):
        for j, c in enumerate(COUNTRIES):
            r = Rb[(Rb.spec == sp) & (Rb.outcome == o) & (Rb.predictor == p) & (Rb.country == c)]
            if len(r):
                ax.plot(r.beta.iloc[0], i + (-0.15 if j == 0 else 0.15), "o", ms=3.6, color=COL[c],
                        mfc=COL[c] if r.p.iloc[0] < .05 else "white")
    ax.axvline(0, color="black", lw=0.4); ax.set_title(ttl, fontsize=6.8, loc="left"); ax.tick_params(labelsize=6)
axes[0].set_yticks(range(len(specs))); axes[0].set_yticklabels(specs, fontsize=6.2); axes[0].invert_yaxis()
for ax in axes:
    ax.set_xlabel("Std. β", fontsize=6.5)
handles = [Line2D([], [], color=COL[c], marker="o", ms=4, ls="", label=c) for c in COUNTRIES] + \
          [Line2D([], [], color="grey", marker="o", ms=4, mfc="white", ls="", label="not significant (p ≥ .05)")]
fig.legend(handles=handles, loc="outside lower center", ncol=3)
savefig(fig, "FigS2_sem_robustness")

In [ ]:
# Figure S3: how the summed SHAP of the 14 harm items changes with the mean harm score
dep = pd.read_csv(TAB / "T9d_shap_harm_dependence.csv")
fig, axes = plt.subplots(1, 4, figsize=(7.2, 2.6), sharey=True, layout="constrained")
for ax, o in zip(axes, [OUTCOMES[k] for k in SHAP_OUT]):
    for cty in COUNTRIES:
        q = dep[(dep.outcome == o) & (dep.country == cty)].dropna()
        ax.scatter(q.harm_mean + np.random.default_rng(1).normal(0, 0.03, len(q)), q.harm_shap_sum, s=3, alpha=0.35,
                   color=COL[cty], lw=0, label=cty)
        sm_ = lowess(q.harm_shap_sum, q.harm_mean, frac=0.4)
        ax.plot(sm_[:, 0], sm_[:, 1], color=COL[cty], lw=1.3)
    ax.axhline(0, color="black", lw=0.4); ax.set_title(o, fontsize=7, loc="left"); ax.set_xlabel("Mean harm score (1-5)", fontsize=6.5)
axes[0].set_ylabel("Summed SHAP of harm items", fontsize=6.5)
fig.legend(*axes[0].get_legend_handles_labels(), loc="outside lower center", ncol=2, markerscale=3)
savefig(fig, "FigS3_shap_harm_dependence")

In [ ]:
# Figure S4: the ten most important single features per instrument and country
fig, axes = plt.subplots(4, 2, figsize=(7.2, 9.8), layout="constrained")
for i, o in enumerate([OUTCOMES[k] for k in SHAP_OUT]):
    for j, cty in enumerate(COUNTRIES):
        ax = axes[i, j]
        s_ = T9a[(T9a.outcome == o) & (T9a.country == cty)].sort_values("mean_abs_shap", ascending=False).head(10)[::-1]
        ax.errorbar(s_.mean_abs_shap, range(10), xerr=[s_.mean_abs_shap - s_.lo, s_.hi - s_.mean_abs_shap], fmt="o", ms=3,
                    color=COL[cty], elinewidth=0.7, capsize=1)
        ax.set_yticks(range(10)); ax.set_yticklabels(s_.label, fontsize=6)
        ax.set_title(f"{'abcdefgh'[2 * i + j]}) {o}: {cty}", fontsize=7, loc="left"); ax.set_xlim(0, max(0.8, s_.hi.max() * 1.05))
        if i == 3:
            ax.set_xlabel("Mean |SHAP| (log-odds), 95% bootstrap interval", fontsize=6.5)
xmax = max(ax.get_xlim()[1] for ax in axes.flat)          # same x-axis in every panel
for ax in axes.flat:
    ax.set_xlim(0, xmax); ax.set_xticks(np.arange(0, xmax, 0.2))
savefig(fig, "FigS4_shap_top_features")

## Step 8. Save every table in one Excel workbook

In [ ]:
from PIL import Image
files = sorted(TAB.glob("T*.csv"), key=lambda p: (len(p.stem.split("_")[0]), p.stem))
with pd.ExcelWriter(ROOT / "outputs" / "All_tables.xlsx") as xw:
    pd.DataFrame({"sheet": [f.stem[:31] for f in files], "file": [f.name for f in files]}).to_excel(xw, sheet_name="INDEX", index=False)
    for f in files:
        pd.read_csv(f).to_excel(xw, sheet_name=f.stem[:31], index=False)
manifest = pd.DataFrame([{"figure": f.name, "width_px": Image.open(f).size[0], "height_px": Image.open(f).size[1],
                          "dpi": round(Image.open(f).info["dpi"][0])} for f in sorted(FIG.glob("*.png"))])
manifest.to_csv(ROOT / "outputs" / "figure_manifest.csv", index=False)
print(len(files), "tables saved to outputs/All_tables.xlsx")
manifest

## Optional: inter-coder agreement for the open-ended codes

Step 6 saved 120 answers to `docs/text_validation_sample_for_coders.csv`. Two people code each answer by hand: put 1 (theme present) or 0 (absent) in every `coder1::` and `coder2::` column, then run the cell below. It reports Cohen's kappa between the two coders and between the codebook and each coder. Until the columns are filled, the cell only prints a reminder.

In [ ]:
from sklearn.metrics import cohen_kappa_score
v = pd.read_csv(ROOT / "docs" / "text_validation_sample_for_coders.csv")
themes = [c.split("::")[1] for c in v.columns if c.startswith("auto::")]
if v[[f"coder1::{c}" for c in themes] + [f"coder2::{c}" for c in themes]].isna().all().all():
    print("Coder columns are still empty. Fill them with 0/1 and run this cell again.")
else:
    rows = []
    for c in themes:
        a, h1, h2 = v[f"auto::{c}"], v[f"coder1::{c}"], v[f"coder2::{c}"]
        ok = h1.notna() & h2.notna()
        k = lambda x, y: cohen_kappa_score(x[ok].astype(int), y[ok].astype(int)) if x[ok].nunique() + y[ok].nunique() > 2 else np.nan
        rows.append({"theme": c, "n": int(ok.sum()), "kappa_coder1_vs_coder2": k(h1, h2),
                     "kappa_auto_vs_coder1": k(a, h1), "kappa_auto_vs_coder2": k(a, h2)})
    T11e = pd.DataFrame(rows).round(3)
    T11e.to_csv(TAB / "T11e_intercoder_agreement.csv", index=False)
    display(T11e)

## Done

* Merged data: `data/processed/survey_merged_deidentified.csv` (and `.xlsx`).
* Tables: `outputs/tables/` and `outputs/All_tables.xlsx`.
* Figures: `outputs/figures/` (PNG and TIFF at 600 dpi, plus PDF).
* R logs and full model summaries: `outputs/logs/`, `outputs/models/`.